# Setup NemoClaw

This notebook is meant to run **on the host itself** — a [Brev](https://brev.nvidia.com) instance or any other Linux host. It walks through the host-side flow for **NemoClaw**: create or recreate a sandbox, pin the tested Docker package versions, install NemoClaw and onboard the sandbox with your chosen agent harness (OpenClaw or Hermes), apply the VSS policy (the VSS skills and workspace bootstrap docs are baked into the sandbox image, built from the harness's Dockerfile under `agent-harness/`), optionally register an HTTPS VSS Orchestrator MCP path (only when `ORCHESTRATOR_ENABLE_HTTPS` is `True`; the default HTTP path is started later by `deploy_vss_orchestrator.ipynb` and needs no registration here), configure optional OpenClaw webhooks, then optionally verify the live sandbox, active policy, webhooks, and installed workspace docs.

A few steps are Brev-specific — the notebook reads secure-link FQDNs from `BREV_ENVIRONMENT_CONTEXT_PATH` (default `/etc/brev/environment-context.json`) and the generated remote UI link — and are called out where they apply. On other platforms, install the prerequisites yourself and reach the agent UI over your own networking (e.g. the SSH tunnel shown in section 3.5).

Once NemoClaw is up (and you have opened the Agent UI in section 3.5), optionally run **`deploy_nemo_relay.ipynb`** to add trajectory capture to an OpenClaw sandbox. Run it again after recreating the sandbox. Then continue with **`deploy_vss_orchestrator.ipynb`** to prepare the host, start the VSS Orchestrator MCP server, and deploy/manage VSS from the agent UI.

**Required prerequisites**

- Run this notebook with **Python 3.11 or newer**.
- Make sure the intended VSS checkout is the one resolved by `VSS_REPO_DIR`. By default this notebook uses `~/video-search-and-summarization`; set the `VSS_REPO_DIR` environment variable before launching Jupyter if your checkout lives elsewhere or the host has multiple clones.
- If you are re-onboarding a host that previously ran an older OpenShell gateway, follow the NemoClaw sandbox lifecycle upgrade guidance: <https://docs.nvidia.com/nemoclaw/manage-sandboxes/lifecycle>.

**What this notebook covers**

- Choose an agent model provider (which sets the required API key, when that provider needs one) and set notebook options.
- Run preflight checks for the local repo, scripts, policy file, and host prerequisites, then pin Docker to the tested package versions.
- Create or reuse the NemoClaw sandbox, built from the repo's harness image (`.openclaw/Dockerfile` or `.hermes/Dockerfile`, which carry the VSS skills, workspace docs and `vss` CLI), configure the chosen agent model provider, apply the VSS policy, optionally register HTTPS VSS Orchestrator MCP access when enabled, and bake the agent UI origin via `CHAT_UI_URL` at onboard.
- Open the Agent UI (OpenClaw shown as an example); the companion notebook then verifies the agent and deploys VSS.
- *(Optional)* Verify the live sandbox state, active policy metadata, OpenClaw webhooks (if enabled), and installed workspace docs.

**Security:** prefer `NVIDIA_API_KEY` from environment variables or your platform's secret store (e.g. Brev secrets). Do **not** commit notebook outputs that contain credentials or live access tokens.


## 1. Settings

Configure the notebook in two parts:

1. **Initialize provider variables** — seed the agent model-provider variables before you choose a provider.
2. **Choose ONE agent model provider** — pick exactly one of (a) an OpenAI-compatible endpoint you already run, cloud or self-hosted, (b) a local model NemoClaw installs and manages for you, or (c) a model from build.nvidia.com. A model router is (a) with a route id.

> Run section 1.1, then run **exactly one** of the (a)–(c) cells. Section 1.3 holds advanced defaults you can usually leave alone.

<span style="color:red"><strong>Important:</strong> set at least one of <code>NVIDIA_API_KEY</code> / <code>COMPATIBLE_API_KEY</code> via the provider cell you pick when that provider needs a key. Credentials can also come from the environment or your platform's secret store (e.g. Brev secrets).</span>

### 1.1 Initialize agent model-provider variables

Run this cell first. It seeds the agent model-provider variables to empty so that whichever **one** of the (a)–(c) cells you run in section 1.2 works on its own.


In [ ]:
# ================== Agent model vars (set by ONE of (a)-(c) in 1.2) ==================
NVIDIA_API_KEY = NEMOCLAW_ENDPOINT_URL = NEMOCLAW_MODEL = COMPATIBLE_API_KEY = NEMOCLAW_PROVIDER = ""

### 1.2 Choose ONE agent model provider

Run **exactly one** of the cells below. 

| Option | When to use | Provider | Also sets |
|---|---|---|---|
| **(a) OpenAI-compatible endpoint** | An endpoint that already exists — a SOTA cloud model (Claude Opus, GPT-5, …) for best agent quality, a server you host yourself, or a model router, which takes a route id in place of a model id. | `custom` | `NEMOCLAW_ENDPOINT_URL` and `NEMOCLAW_MODEL`, both required; `COMPATIBLE_API_KEY` required for a public endpoint, optional for a self-hosted one |
| **(b) NemoClaw-managed local model** | Self-hosted on this box, air-gapped — NemoClaw installs and starts the server for you. | `install-vllm`, `ollama`, `nim-local`, … | *`install-vllm`:* `NEMOCLAW_MODEL` optional — a serving-catalog slug, blank takes the catalog default<br>*`ollama` / `nim-local`:* `NEMOCLAW_MODEL` optional — blank uses the host default |
| **(c) build.nvidia.com NVIDIA-hosted model** | Zero-setup — uses NVIDIA's hosted Nemotron via `integrate.api.nvidia.com`, and needs only an NVIDIA API key. | `build` | `NVIDIA_API_KEY`, optionally `NEMOCLAW_MODEL` |


#### (a) OpenAI-compatible endpoint you already run — *recommended for best agent quality*

Any endpoint that speaks the OpenAI chat-completions API, cloud or self-hosted. Use (b) instead when you want NemoClaw to install and manage the server.

`COMPATIBLE_API_KEY` is the one value the cell does not prefill. For the prefilled Inference Hub endpoint, create a key at <https://inference.nvidia.com/key-management?action=new-key>; the model ids it serves are listed at <https://inference.nvidia.com/?new=0>. For any other endpoint, see that provider's own documentation for its key and model ids. Prefer exporting the key from your shell or platform secret store over typing it into the cell: left blank here, 1.3 takes it from `COMPATIBLE_API_KEY` in the environment the kernel started in, so restart the kernel after changing it.

Section 3.1 picks the transport from the endpoint's address, not from which cell you ran. A host resolving to a private address gets the bundled host proxy, so NemoClaw's SSRF guard accepts it, and a blank `COMPATIBLE_API_KEY` is sent as an `EMPTY` placeholder — a self-hosted server ignores the value, and the installer rejects a truly empty one. A public endpoint needs its real bearer token.

A **model router** is one of these endpoints: give its address here and put the
route id it serves in `NEMOCLAW_MODEL` instead of a model id. Start it first with
[`deploy_vss_switchyard.ipynb`](deploy_vss_switchyard.ipynb), which builds it from a
pinned source ref and verifies traffic really does split across targets. The router
forwards the caller's own credential upstream and stores nothing, so put your real
upstream key in `COMPATIBLE_API_KEY`.

The sandbox's egress to a router is granted per deployment rather than shipped open:
that notebook writes a `vss-model-router` preset next to the router config, and
section 3.2 below applies it whenever the file exists, including after a re-onboard.
A deployment that does not route never opens the port. It also writes the matching
revoke preset and prints the order to close it again.

> Self-hosted: the sandbox reaches this box as `host.openshell.internal` — rebind a loopback-only server to `0.0.0.0` or it stays unreachable.



In [ ]:
# (a) OpenAI-compatible endpoint — add your key and run. Endpoint and model are
#     prefilled with the default, Claude Opus 5 via the NVIDIA Inference Hub,
#     which is reachable from NVIDIA infrastructure. Replace both for any other
#     endpoint: a provider's public API ("https://api.anthropic.com/v1/") serves
#     the bare "claude-opus-5", self-hosted is "http://host.openshell.internal:8000/v1",
#     and a router takes the route id it serves, e.g. "switchyard/random".
NEMOCLAW_PROVIDER     = "custom"
NEMOCLAW_ENDPOINT_URL = "https://inference-api.nvidia.com/v1"  # OpenAI-compatible base URL
NEMOCLAW_MODEL        = "aws/anthropic/bedrock-claude-opus-5"  # Model id at that endpoint
COMPATIBLE_API_KEY    = ""  # Bearer token (sk-ant-..., sk-proj-...); blank takes the exported one, and is fine for a self-hosted server

#### (b) NemoClaw-managed local model — *self-hosted / air-gapped*

NemoClaw installs and starts the server itself. Set `NEMOCLAW_PROVIDER` to one of its managed providers (`install-vllm`, `ollama`, `nim-local`, …). For `install-vllm`, `NEMOCLAW_MODEL` takes a serving-catalog slug (`nemotron-3.5-lightning-30b`, `muse-glimmer-30b`, `deepseek-r1-distill-70b`, …) which 3.1 passes to onboard as `NEMOCLAW_VLLM_MODEL`; leave it blank to take the catalog default for this host. An unknown slug fails onboard up front, before it touches Docker or the sandbox. For `ollama` / `nim-local`, `NEMOCLAW_MODEL` is optional (blank uses the host default).

On a multi-GPU box, `install-vllm` otherwise takes every visible GPU: set `NEMOCLAW_VLLM_GPU_DEVICE` in 1.3 to leave the rest for the VSS deployment.

> Already running your own OpenAI-compatible server — a downloadable NIM, a vLLM you started, Ollama on this box? Use (a) and give it the endpoint URL. Nothing in this cell applies to a server NemoClaw does not manage.

In [ ]:
# (b) NemoClaw-managed local model — pick the provider, then fill the fields it needs.

NEMOCLAW_PROVIDER = "install-vllm"  # install-vllm / ollama / nim-local, ...
NEMOCLAW_MODEL = "nemotron-3.5-lightning-30b"  # e.g. "nemotron-3.5-lightning-30b / muse-glimmer-30b"

# install-vllm only — required for a gated model (deepseek-r1-distill-70b); otherwise optional.
HF_TOKEN = ""  # Hugging Face read token, "hf_..."

# Self-clearing only, do not set.
NEMOCLAW_ENDPOINT_URL = ""
COMPATIBLE_API_KEY    = ""


#### (c) build.nvidia.com NVIDIA-hosted model — *zero-setup, NVIDIA key only*

Uses NVIDIA's hosted model catalog via `integrate.api.nvidia.com`. Only `NVIDIA_API_KEY` is required; leave `NEMOCLAW_MODEL` blank to use the NemoClaw default.

> Get a key at <https://build.nvidia.com> (format `nvapi-...`). To use a different hosted model, set `NEMOCLAW_MODEL` to its build.nvidia.com id (e.g. `nvidia/llama-3.3-nemotron-super-49b-v1.5`).


In [ ]:
# (c) build.nvidia.com — set NVIDIA_API_KEY, then run.
NEMOCLAW_PROVIDER = "build"
NVIDIA_API_KEY = ""  # "nvapi-..." from https://build.nvidia.com
NEMOCLAW_MODEL = "nvidia/nemotron-3.5-lightning-30b-a3b"  # leave blank to use the NemoClaw default, or set a build.nvidia.com model id

# Self-clearing only, do not set.
NEMOCLAW_ENDPOINT_URL = ""
COMPATIBLE_API_KEY    = ""

### 1.3 Advanced settings (defaults — usually leave alone)

- `NEMOCLAW_INSTALL_REF` — pinned installer.
- `AGENT_RUNTIME` — `openclaw` or `hermes` (env override). Add an `AGENT_HARNESS_PROFILES` entry for a new harness.
- `AGENT_IMAGE_DOCKERFILE` — the repo Dockerfile onboard builds the sandbox from (`onboard --from`), per harness in `AGENT_HARNESS_PROFILES`: `.openclaw/Dockerfile` (VSS OpenClaw plugin: skills selected per deployment, workspace docs, `vss` CLI) or `.hermes/Dockerfile` (skills, docs, `vss` CLI). Env override for a local variant.
- `NEMOCLAW_INFERENCE_PROXY` — SSRF check and host proxy for local endpoints (env override; `0` disables it). The proxy reaches its upstream over `https` on port 443, so turn it off for a self-hosted endpoint served over plain HTTP or on another port — such as this deployment's own LLM NIM.
- `AGENT_HOOKS_*` / `AGENT_WEBHOOK_PORT` — inbound webhooks (Hermes default `8644`).
- `AGENT_DASHBOARD_PORT` — derived (`18789`; override via `NEMOCLAW_DASHBOARD_PORT`).
- `HITL_ENABLED` — structured mid-run questions. Keep `False` for the launch configuration so the agent asks follow-ups as ordinary chat messages.
- `NEMOCLAW_VLLM_PORT` — host port for NemoClaw-managed vLLM. Used by `install-vllm`
- `NEMOCLAW_VLLM_GPU_DEVICE` — which GPU the managed vLLM gets: index (`1`) or full GPU/MIG UUID from `nvidia-smi -L`. Blank keeps the provider default (all visible GPUs). Passed as `onboard --vllm-gpu-device` when the provider is `install-vllm` and dropped otherwise, so a change needs a fresh onboard. It is also dropped when a vLLM is already serving on `NEMOCLAW_VLLM_PORT`, since NemoClaw attaches to that one. Section 3 checks it against `nvidia-smi -L` in the runs that do pass it, instead of leaving `nemoclaw` to reject it partway through the onboard.
- `NEMOCLAW_TOOL_DISCLOSURE` — `direct` or `progressive` (env override). Blank keeps NemoClaw's default, except Nemotron 3.5 Lightning defaults to `direct`: that model does not reliably complete OpenClaw's experimental search/describe/call bridge, while direct structured tool calls work. A change needs a sandbox rebuild or fresh onboard.
- `BREV_ENVIRONMENT_CONTEXT_PATH` — derived (`/etc/brev/environment-context.json`).
- `ORCHESTRATOR_ENABLE_HTTPS` — `False` (default) skips 3.3; the agent uses the HTTP orchestrator from `deploy_vss_orchestrator.ipynb`. `True` registers the HTTPS MCP URL. Set the same value in both notebooks.
- `NEMOCLAW_RECREATE_SANDBOX` — `True` (default) re-onboards an existing sandbox in 3.1 with `--recreate-sandbox`, discarding it and its agent sessions; 3.2–3.4 then reapply the policy and the rest of the setup. `False` reuses it and skips onboard. Shell override: `1` or `0`.

Every setting here can be overridden from the shell that launches the notebook (`NEMOCLAW_INSTALL_REF=v0.0.115 jupyter nbconvert --execute …`, papermill, platform secrets). Overrides are read from `SHELL_ENV`, a snapshot of the environment taken the first time this cell runs in a kernel, so re-running the cell after editing a value above picks up the edit instead of the variables 3.1 exported on the previous run. Restart the kernel to pick up a shell override changed after the notebook was opened.


In [ ]:
import os
import subprocess
from pathlib import Path


# ================== Default Nemoclaw settings ==================
# NemoClaw installer pin. Must ship the public sandbox-first lifecycle
# commands used below (`{cli} {sandbox} policy-add|skill install|mcp|…`).
NEMOCLAW_INSTALL_REF = "v0.0.114"
# Sandbox agent harness: "openclaw" or "hermes". Override via the AGENT_RUNTIME env var.
# To add a new harness, extend AGENT_HARNESS_PROFILES below — downstream cells read the profile.
AGENT_RUNTIME = "openclaw"
# Enable SSRF detection for NEMOCLAW_ENDPOINT_URL and host proxy startup.
NEMOCLAW_INFERENCE_PROXY = True
AGENT_HOOKS_ENABLED = True     # enable agent inbound webhooks when the harness supports them
AGENT_HOOKS_PATH = "/hooks"     # OpenClaw hooks path (ignored for Hermes)
AGENT_WEBHOOK_PORT = 8644       # Hermes webhook adapter port (ignored for OpenClaw)
HITL_ENABLED = False            # False => ask follow-ups in ordinary chat turns
NEMOCLAW_VLLM_PORT = 18000     # Host port for NemoClaw-hosted vLLM
# GPU for NemoClaw-hosted vLLM: an index ("1") or a full GPU/MIG UUID.  Blank leaves the provider's default (all
# visible GPUs).
NEMOCLAW_VLLM_GPU_DEVICE = "0"     # 0-based index (0 = first GPU)
# Re-onboard an existing sandbox in 3.1 with --recreate-sandbox, replacing it; False
# reuses it and skips onboard.
NEMOCLAW_RECREATE_SANDBOX = True
# Must match ORCHESTRATOR_ENABLE_HTTPS in deploy_vss_orchestrator.ipynb.
ORCHESTRATOR_ENABLE_HTTPS = False
# Ingress origin of the Kubernetes VSS deployment this sandbox should operate,
# e.g. "http://vss-search.<node-ip>.nip.io". Fills both the egress policy's
# vss-k8s-ingress host (3.2) and ENV.md's export line (also 3.2). Leave empty for
# Compose sandboxes, or when the deployment does not exist yet: the agent
# then asks the user for it in chat.
VSS_PUBLIC_URL = ""


# ================== Derived (no need to touch) ==================

SHELL_ENV = globals().setdefault("_NOTEBOOK_SHELL_ENV", dict(os.environ))

HOME_DIR = Path.home().resolve()
NVIDIA_API_KEY = (NVIDIA_API_KEY or SHELL_ENV.get("NVIDIA_API_KEY", "")).strip()
COMPATIBLE_API_KEY = (COMPATIBLE_API_KEY or SHELL_ENV.get("COMPATIBLE_API_KEY", "")).strip()
# Brev context JSON (secure-link FQDNs + environment id). Override via env if needed.
BREV_ENVIRONMENT_CONTEXT_PATH = SHELL_ENV.get("BREV_ENVIRONMENT_CONTEXT_PATH", "/etc/brev/environment-context.json").strip()
os.environ["BREV_ENVIRONMENT_CONTEXT_PATH"] = BREV_ENVIRONMENT_CONTEXT_PATH
# Agent UI port (OpenClaw / Hermes). Override via NEMOCLAW_DASHBOARD_PORT if needed.
AGENT_DASHBOARD_PORT = int(SHELL_ENV.get("NEMOCLAW_DASHBOARD_PORT", "18789") or "18789")
VSS_REPO_DIR = Path(SHELL_ENV.get("VSS_REPO_DIR", HOME_DIR / "video-search-and-summarization")).resolve()
NEMOCLAW_REPO_DIR = Path(SHELL_ENV.get("NEMOCLAW_REPO_DIR", HOME_DIR / "NemoClaw")).resolve()
INFERENCE_API_PROXY_PORT = int(SHELL_ENV.get("INFERENCE_API_PROXY_PORT", "18080"))
# Read here rather than through run_setup_notebook's parameter table: that table
# assigns the environment string, and 3.1 rejects anything but a real bool.
NEMOCLAW_INFERENCE_PROXY = (
    SHELL_ENV.get("NEMOCLAW_INFERENCE_PROXY", "").strip().lower()
    or ("1" if NEMOCLAW_INFERENCE_PROXY else "0")
) not in ("0", "false", "no", "off")
_hitl_enabled_raw = (
    SHELL_ENV.get("HITL_ENABLED", "").strip().lower()
    or ("true" if HITL_ENABLED else "false")
)
if _hitl_enabled_raw not in ("1", "true", "yes", "on", "0", "false", "no", "off"):
    raise ValueError("HITL_ENABLED must be true or false")
HITL_ENABLED = _hitl_enabled_raw in ("1", "true", "yes", "on")
AGENT_RUNTIME = (SHELL_ENV.get("AGENT_RUNTIME") or AGENT_RUNTIME).strip().lower()

# Fill `{cli}` here once the harness is chosen; later cells still
# `.format(sandbox=..., path=..., …)` the remaining placeholders.
def _with_cli(tmpl: str, cli: str) -> str:
    return tmpl.replace("{cli}", cli)


_AGENT_SANDBOX_CMDS = {
    "policy_add_cmd": "{cli} {sandbox} policy-add --from-file {path} --yes",
    "upload_cmd": "{cli} {sandbox} upload {doc} {dest}/",
    "mcp_status_cmd": "{cli} {sandbox} mcp status {server}",
    "mcp_add_cmd": "{cli} {sandbox} mcp add {server} --url {url}",
    "mcp_remove_cmd": "{cli} {sandbox} mcp remove {server}",
    "config_set_cmd": "{cli} {sandbox} config set --key {key} --value {value} --config-accept-new-path",
    "gateway_restart_cmd": "{cli} {sandbox} gateway restart",
    "recover_cmd": "{cli} {sandbox} recover",
    "gateway_token_cmd": "{cli} {sandbox} gateway-token --quiet",
    "dashboard_url_cmd": "{cli} {sandbox} dashboard-url --quiet",
    "connect_cmd": "{cli} {sandbox} connect",
}

# Per-harness differences only (cli, onboard, paths, UI, verify).
AGENT_HARNESS_PROFILES = {
    "openclaw": {
        "label": "NemoClaw",
        "cli": "nemoclaw",
        "onboard_args": "--non-interactive --agent openclaw",
        # Sandbox image built by onboard (`--from`), relative to VSS_REPO_DIR. Every
        # harness image carries the VSS skills, the workspace docs and the `vss`
        # CLI; the OpenClaw one as a plugin that selects skills per deployment.
        "image_dockerfile": ".openclaw/Dockerfile",
        "retry_onboard_on_fail": True,
        "workspace_remote_dir": "/sandbox/.openclaw/workspace",
        "ui_mode": "gateway_token",  # control UI via #token= from gateway-token
        "verify_kind": "openclaw_workspace",
    },
    "hermes": {
        "label": "NemoHermes",
        "cli": "nemohermes",
        "onboard_args": "--non-interactive",
        "image_dockerfile": ".hermes/Dockerfile",
        "retry_onboard_on_fail": True,
        "workspace_remote_dir": "/sandbox",
        "ui_mode": "dashboard_url",  # plain URL; gateway-token is the :8642 API bearer
        "verify_kind": "sandbox_docs",
        "verify_cmds": (
            ("{label} top-level docs", "ls -1 {workspace_remote_dir}/*.md 2>/dev/null || true"),
            (
                "VSS Orchestrator MCP host alias",
                "curl -s -o /dev/null --max-time 5 {scheme}://{host_alias}:{mcp_port}/ && echo host alias reachable || true",
            ),
        ),
    },
}
AGENT_HARNESS = AGENT_HARNESS_PROFILES.get(AGENT_RUNTIME)
if AGENT_HARNESS is None:
    known = ", ".join(sorted(AGENT_HARNESS_PROFILES))
    raise KeyError(f"Unknown AGENT_RUNTIME={AGENT_RUNTIME!r}; known harnesses: {known}")

AGENT_LABEL = AGENT_HARNESS["label"]
AGENT_CLI = AGENT_HARNESS["cli"]
_onboard_args = AGENT_HARNESS["onboard_args"].strip()
AGENT_ONBOARD_CMD = f"{AGENT_CLI} onboard {_onboard_args}"
AGENT_RETRY_ONBOARD_ON_FAIL = bool(AGENT_HARNESS["retry_onboard_on_fail"])
_image_dockerfile = (SHELL_ENV.get("AGENT_IMAGE_DOCKERFILE") or AGENT_HARNESS.get("image_dockerfile") or "").strip()
if not _image_dockerfile:
    raise KeyError(f"AGENT_HARNESS_PROFILES[{AGENT_RUNTIME!r}] has no image_dockerfile; every harness is built from its .<name>/Dockerfile")
WORKSPACE_REMOTE_DIR = AGENT_HARNESS["workspace_remote_dir"]
_ui_mode = AGENT_HARNESS.get("ui_mode") or "dashboard_url"
AGENT_UI_USES_GATEWAY_TOKEN = _ui_mode == "gateway_token"
AGENT_POLICY_ADD_CMD = _with_cli(_AGENT_SANDBOX_CMDS["policy_add_cmd"], AGENT_CLI)
AGENT_UPLOAD_CMD = _with_cli(_AGENT_SANDBOX_CMDS["upload_cmd"], AGENT_CLI)
AGENT_MCP_STATUS_CMD = _with_cli(_AGENT_SANDBOX_CMDS["mcp_status_cmd"], AGENT_CLI)
AGENT_MCP_ADD_CMD = _with_cli(_AGENT_SANDBOX_CMDS["mcp_add_cmd"], AGENT_CLI)
AGENT_MCP_REMOVE_CMD = _with_cli(_AGENT_SANDBOX_CMDS["mcp_remove_cmd"], AGENT_CLI)
AGENT_CONFIG_SET_CMD = _with_cli(_AGENT_SANDBOX_CMDS["config_set_cmd"], AGENT_CLI)
AGENT_GATEWAY_RESTART_CMD = _with_cli(_AGENT_SANDBOX_CMDS["gateway_restart_cmd"], AGENT_CLI)
AGENT_RECOVER_CMD = _with_cli(_AGENT_SANDBOX_CMDS["recover_cmd"], AGENT_CLI)
AGENT_CONNECT_CMD = _with_cli(_AGENT_SANDBOX_CMDS["connect_cmd"], AGENT_CLI)
AGENT_GATEWAY_TOKEN_CMD = (
    _with_cli(_AGENT_SANDBOX_CMDS["gateway_token_cmd"], AGENT_CLI)
    if AGENT_UI_USES_GATEWAY_TOKEN
    else None
)
AGENT_DASHBOARD_URL_CMD = (
    _with_cli(_AGENT_SANDBOX_CMDS["dashboard_url_cmd"], AGENT_CLI)
    if _ui_mode == "dashboard_url"
    else None
)
AGENT_VERIFY_KIND = AGENT_HARNESS["verify_kind"]
AGENT_VERIFY_CMDS = AGENT_HARNESS.get("verify_cmds") or ()


DEPLOY_SCRIPTS_DIR = VSS_REPO_DIR / "deploy" / "docker" / "scripts"
NEMOCLAW_PROVIDER = (NEMOCLAW_PROVIDER or SHELL_ENV.get("NEMOCLAW_PROVIDER", "")).strip()
if not NEMOCLAW_PROVIDER:
    NEMOCLAW_PROVIDER = "custom" if NEMOCLAW_ENDPOINT_URL else "build"
NEMOCLAW_TOOL_DISCLOSURE = SHELL_ENV.get("NEMOCLAW_TOOL_DISCLOSURE", "").strip().lower()
if not NEMOCLAW_TOOL_DISCLOSURE and AGENT_RUNTIME == "openclaw" and "nemotron-3.5-lightning" in NEMOCLAW_MODEL.lower():
    NEMOCLAW_TOOL_DISCLOSURE = "direct"
if NEMOCLAW_TOOL_DISCLOSURE not in ("", "direct", "progressive"):
    raise ValueError("NEMOCLAW_TOOL_DISCLOSURE must be direct, progressive, or blank.")
if NEMOCLAW_TOOL_DISCLOSURE:
    os.environ["NEMOCLAW_TOOL_DISCLOSURE"] = NEMOCLAW_TOOL_DISCLOSURE
else:
    os.environ.pop("NEMOCLAW_TOOL_DISCLOSURE", None)
POLICY_PATH = VSS_REPO_DIR / "assets" / "vss_nemoclaw_policy.yaml"
SKILLS_DIR = VSS_REPO_DIR / "skills"
WORKSPACE_VARIANT = SHELL_ENV.get("AGENT_PLUGIN_VARIANT", "nemoclaw").strip() or "nemoclaw"
WORKSPACE_DIR = (VSS_REPO_DIR / ".openclaw" / "workspace").resolve()
# Resolved after VSS_REPO_DIR: the Dockerfile's parent directory is the build context.
AGENT_IMAGE_DOCKERFILE = (VSS_REPO_DIR / _image_dockerfile).resolve()
SANDBOX_CONFIG_PATH = "/sandbox/.openclaw/openclaw.json"
AGENT_HOOKS_TOKEN = subprocess.check_output(["openssl", "rand", "-hex", "32"], text=True).strip() if AGENT_HOOKS_ENABLED else ""
INFERENCE_API_PROXY_PATH = DEPLOY_SCRIPTS_DIR / "nemoclaw" / "inference-api-proxy.py"
ORCHESTRATOR_MCP_HELPER_PATH = DEPLOY_SCRIPTS_DIR / "orchestrator_mcp_helper.py"
BREV_UTIL_PATH = VSS_REPO_DIR / "services" / "agent" / "packages" / "vss_agents" / "src" / "vss_agents" / "orchestrator" / "brev_util.py"
MCP_PORT = int(SHELL_ENV.get("VSS_ORCHESTRATOR_MCP_PORT", "9988"))
HOST_INTERNAL_ALIAS = SHELL_ENV.get("HOST_INTERNAL_ALIAS", "host.openshell.internal").strip()
ORCHESTRATOR_ENABLE_HTTPS = (SHELL_ENV.get("ORCHESTRATOR_ENABLE_HTTPS", str(ORCHESTRATOR_ENABLE_HTTPS)).strip().lower() == "true")
MCP_SCHEME = "https" if ORCHESTRATOR_ENABLE_HTTPS else "http"
ORCHESTRATOR_MCP_SERVER = SHELL_ENV.get("ORCHESTRATOR_MCP_SERVER", "vss_orchestrator").strip() or "vss_orchestrator"
ORCHESTRATOR_MCP_URL = f"{MCP_SCHEME}://{HOST_INTERNAL_ALIAS}:{MCP_PORT}/mcp"
NEMOCLAW_SANDBOX_NAME = SHELL_ENV.get("NEMOCLAW_SANDBOX_NAME", "demo").strip()
NEMOCLAW_INSTALL_REF = SHELL_ENV.get("NEMOCLAW_INSTALL_REF", NEMOCLAW_INSTALL_REF).strip()
# Checked and normalized in 3.2, next to the first cell that hands it to
# nemoclaw; 3.4 re-runs the same check before writing it into ENV.md.
VSS_PUBLIC_URL = (SHELL_ENV.get("VSS_PUBLIC_URL") or VSS_PUBLIC_URL).strip().rstrip("/")
AGENT_WEBHOOK_PORT = int(SHELL_ENV.get("AGENT_WEBHOOK_PORT", str(AGENT_WEBHOOK_PORT)) or "8644")
NEMOCLAW_VLLM_PORT = int(SHELL_ENV.get("NEMOCLAW_VLLM_PORT", str(NEMOCLAW_VLLM_PORT)) or "18000")
if not 1024 <= NEMOCLAW_VLLM_PORT <= 65535:
    raise ValueError(f"NEMOCLAW_VLLM_PORT={NEMOCLAW_VLLM_PORT} is outside 1024-65535.")
NEMOCLAW_VLLM_GPU_DEVICE = str(SHELL_ENV.get("NEMOCLAW_VLLM_GPU_DEVICE", NEMOCLAW_VLLM_GPU_DEVICE)).strip()
# Only install-vllm starts a server for NemoClaw to place, and the other providers
# reject the flag. Appended to the onboard calls in 3.1 rather than baked into
# AGENT_ONBOARD_CMD, so 3.1 can drop it for one run when it finds a vLLM already
# serving — NemoClaw attaches to that one instead of installing.
AGENT_VLLM_GPU_FLAG = (
    f" --vllm-gpu-device {NEMOCLAW_VLLM_GPU_DEVICE}"
    if NEMOCLAW_VLLM_GPU_DEVICE and NEMOCLAW_PROVIDER == "install-vllm"
    else ""
)
# "1"/"0", the spelling NemoClaw reads for this same variable name.
NEMOCLAW_RECREATE_SANDBOX = (
    SHELL_ENV.get("NEMOCLAW_RECREATE_SANDBOX", "1" if NEMOCLAW_RECREATE_SANDBOX else "0").strip()
    == "1"
)
print("Sandbox:", NEMOCLAW_SANDBOX_NAME)
print("NEMOCLAW_RECREATE_SANDBOX:", "True" if NEMOCLAW_RECREATE_SANDBOX else "False")
print("NEMOCLAW_INSTALL_REF:", NEMOCLAW_INSTALL_REF)
print("\nHOME_DIR:", HOME_DIR)
print("VSS_REPO_DIR:", VSS_REPO_DIR)
print("NEMOCLAW_REPO_DIR:", NEMOCLAW_REPO_DIR)
print("POLICY_PATH:", POLICY_PATH)
print("SKILLS_DIR:", SKILLS_DIR)
print("WORKSPACE_DIR:", WORKSPACE_DIR)
print("AGENT_IMAGE_DOCKERFILE:", AGENT_IMAGE_DOCKERFILE)
print("WORKSPACE_REMOTE_DIR:", WORKSPACE_REMOTE_DIR)
print("INFERENCE_API_PROXY_PATH:", INFERENCE_API_PROXY_PATH)
print("ORCHESTRATOR_MCP_HELPER_PATH:", ORCHESTRATOR_MCP_HELPER_PATH)
print("BREV_UTIL_PATH:", BREV_UTIL_PATH)
print("AGENT_RUNTIME:", AGENT_RUNTIME)
print("AGENT_LABEL:", AGENT_LABEL)
print("AGENT_CLI:", AGENT_CLI)
print("NEMOCLAW_TOOL_DISCLOSURE:", NEMOCLAW_TOOL_DISCLOSURE or "(NemoClaw default)")
print("BREV_ENVIRONMENT_CONTEXT_PATH:", BREV_ENVIRONMENT_CONTEXT_PATH)
print("AGENT_DASHBOARD_PORT:", AGENT_DASHBOARD_PORT)
print("HITL_ENABLED:", HITL_ENABLED)
print("WORKSPACE_VARIANT:", WORKSPACE_VARIANT)
print("VSS_PUBLIC_URL:", VSS_PUBLIC_URL or "(empty — the agent will ask in chat)")
print("HOST_INTERNAL_ALIAS:", HOST_INTERNAL_ALIAS)
print("ORCHESTRATOR_MCP_SERVER:", ORCHESTRATOR_MCP_SERVER)
print("ORCHESTRATOR_MCP_URL:", ORCHESTRATOR_MCP_URL)
print("ORCHESTRATOR_ENABLE_HTTPS:", ORCHESTRATOR_ENABLE_HTTPS)
print("NEMOCLAW_PROVIDER:", NEMOCLAW_PROVIDER)
print("NEMOCLAW_INFERENCE_PROXY:", NEMOCLAW_INFERENCE_PROXY)
print("NEMOCLAW_VLLM_PORT:", NEMOCLAW_VLLM_PORT)
if not NEMOCLAW_VLLM_GPU_DEVICE:
    print("NEMOCLAW_VLLM_GPU_DEVICE: (blank => provider default)")
elif NEMOCLAW_PROVIDER == "install-vllm":
    print("NEMOCLAW_VLLM_GPU_DEVICE:", NEMOCLAW_VLLM_GPU_DEVICE)
else:
    print(f"NEMOCLAW_VLLM_GPU_DEVICE: {NEMOCLAW_VLLM_GPU_DEVICE} (not passed: provider is {NEMOCLAW_PROVIDER})")
print("NEMOCLAW_MODEL:", NEMOCLAW_MODEL or "(blank => provider default for this host)")
if NEMOCLAW_ENDPOINT_URL:
    print("NEMOCLAW_ENDPOINT_URL:", NEMOCLAW_ENDPOINT_URL)
    print("COMPATIBLE_API_KEY set:", bool(COMPATIBLE_API_KEY))
print("Agent hooks enabled:", AGENT_HOOKS_ENABLED)
if AGENT_HOOKS_ENABLED:
    print("Agent hooks token set:", bool(AGENT_HOOKS_TOKEN))
    if AGENT_RUNTIME == "openclaw":
        print("Agent hooks path:", AGENT_HOOKS_PATH)
    elif AGENT_RUNTIME == "hermes":
        print("Agent webhook port:", AGENT_WEBHOOK_PORT)
print("NVIDIA_API_KEY set:", bool(NVIDIA_API_KEY))

## 2. Preflight

Run the next cell to confirm the expected keys, files, commands are present on the host.


In [ ]:
import importlib.util
import os
import shutil
from pathlib import Path

RED = "\033[31m"
RESET = "\033[0m"
GREEN = "\033[32m"
YELLOW = "\033[33m"

helper_spec = importlib.util.spec_from_file_location("orchestrator_mcp_helper", ORCHESTRATOR_MCP_HELPER_PATH)
if helper_spec is None or helper_spec.loader is None:
    raise ImportError(f"Could not load MCP helper from {ORCHESTRATOR_MCP_HELPER_PATH}")
orchestrator_mcp_helper = importlib.util.module_from_spec(helper_spec)
helper_spec.loader.exec_module(orchestrator_mcp_helper)

brev_util_spec = importlib.util.spec_from_file_location("vss_brev_util", BREV_UTIL_PATH)
if brev_util_spec is None or brev_util_spec.loader is None:
    raise ImportError(f"Could not load brev_util from {BREV_UTIL_PATH}")
brev_util = importlib.util.module_from_spec(brev_util_spec)
brev_util_spec.loader.exec_module(brev_util)

# Needed later: 3.1 (UI origin) and 3.5 (gateway container lookup).
brev_environment_id = brev_util.brev_environment_id
brev_secure_link_fqdn = brev_util.brev_secure_link_fqdn
resolve_openshell_gateway_container = orchestrator_mcp_helper.resolve_openshell_gateway_container


def agent_provider_configured() -> bool:
    if NEMOCLAW_PROVIDER == "build":
        return bool(NVIDIA_API_KEY)
    if NEMOCLAW_PROVIDER == "custom":
        # COMPATIBLE_API_KEY is checked in 3.1, which knows whether the endpoint is
        # self-hosted (blank is fine, it ignores the value) or public (token required).
        return bool(NEMOCLAW_ENDPOINT_URL and NEMOCLAW_MODEL)
    return bool(NEMOCLAW_PROVIDER)


required_checks = {
    "Agent model provider configured": agent_provider_configured(),
    "NEMOCLAW_INSTALL_REF set": bool(NEMOCLAW_INSTALL_REF),
    "vss_nemoclaw_policy.yaml": POLICY_PATH.is_file(),
    "skills/": SKILLS_DIR.is_dir(),
    "workspace bootstrap dir": WORKSPACE_DIR.is_dir(),
    "agent image Dockerfile": AGENT_IMAGE_DOCKERFILE.is_file(),
    "orchestrator_mcp_helper.py": ORCHESTRATOR_MCP_HELPER_PATH.is_file(),
    "brev_util.py": BREV_UTIL_PATH.is_file(),
    # host commands
    "docker": shutil.which("docker") is not None,
    "python3": shutil.which("python3") is not None,
    "curl": shutil.which("curl") is not None,
}

if NEMOCLAW_PROVIDER == "custom":
    required_checks["NEMOCLAW_ENDPOINT_URL set"] = bool(NEMOCLAW_ENDPOINT_URL)
    required_checks["NEMOCLAW_MODEL set"] = bool(NEMOCLAW_MODEL)
elif NEMOCLAW_PROVIDER == "build":
    required_checks["NVIDIA_API_KEY set (build provider)"] = bool(NVIDIA_API_KEY)

if AGENT_HOOKS_ENABLED:
    required_checks["AGENT_HOOKS_TOKEN set"] = bool(AGENT_HOOKS_TOKEN)
    if AGENT_RUNTIME == "openclaw":
        required_checks["AGENT_HOOKS_PATH set"] = bool(AGENT_HOOKS_PATH)
    elif AGENT_RUNTIME == "hermes":
        required_checks["AGENT_WEBHOOK_PORT set"] = bool(AGENT_WEBHOOK_PORT)

for label, ok in required_checks.items():
    status = "OK " if ok else "NO "
    color = GREEN if ok else RED
    print(f"{color}{status}{RESET} {label}")


### 2.1 Pin Docker version

Runs [`pin_docker_version.sh`](pin_docker_version.sh), which pins Docker CE + plugins + containerd.io to the tested combination and `apt-mark hold`s them against drift. That script owns the version numbers and the in-range skip for DGX Spark / DGX-OS on arm64 — read them there.

Run it **before** section 3 brings up the sandbox: a docker-ce downgrade restarts dockerd and would disrupt live sandbox containers. It is idempotent, so on an already-pinned host this only re-applies the holds.

In [ ]:
_pin_script = DEPLOY_SCRIPTS_DIR / "pin_docker_version.sh"
if not _pin_script.is_file():
    raise FileNotFoundError(f"Missing Docker pin script: {_pin_script}")

!bash "{_pin_script}"
assert _exit_code == 0, f"Docker version pin failed: {_pin_script}"

## 3. Install and Configure NemoClaw for VSS skills

Sections 3.1–3.4 install and configure the sandbox using canonical NemoClaw / OpenShell commands. Run the cells in order — 3.2–3.4 are idempotent and safe to re-run, while 3.1 rebuilds the sandbox from scratch on every run unless you set `NEMOCLAW_RECREATE_SANDBOX = False` in 1.3. Section 3.5 opens the Agent UI, and section 3.6 is an optional post-setup verification.

| Step | What it does |
|---|---|
| 3.1 | Install NemoClaw (pinned `NEMOCLAW_INSTALL_REF`) and create the sandbox from the harness image (`.<harness>/Dockerfile`), replacing an existing one by default |
| 3.2 | Apply the VSS sandbox policy and the deployment origin (`ENV.md`) |
| 3.3 | Register the VSS Orchestrator MCP path *(HTTPS only; no-op when `ORCHESTRATOR_ENABLE_HTTPS` is `False`)* |
| 3.4 | Configure optional webhooks |
| 3.5 | Open the Agent UI |
| 3.6 | *(Optional)* Verify sandbox, policy, workspace, and webhooks |


### 3.1 Install NemoClaw and create the sandbox

Exports the NemoClaw configuration (including `CHAT_UI_URL`, the dashboard origin for UI access) and installs NemoClaw at the pinned `NEMOCLAW_INSTALL_REF`. The non-interactive installer also creates and onboards the sandbox. Takes a few minutes; output streams live.

With the `custom` provider, when `NEMOCLAW_ENDPOINT_URL`'s host resolves to a non-public address, this step automatically starts the bundled host proxy and points NemoClaw at it, avoiding NemoClaw's SSRF rejection. Set the advanced setting `NEMOCLAW_INFERENCE_PROXY = False` in 1.3, or export `NEMOCLAW_INFERENCE_PROXY=0`, to disable this behavior; it defaults to `True`. Disable it for an endpoint served over plain HTTP or on a port other than 443 — the proxy always reaches its upstream as `https://<host>` — and pass `COMPATIBLE_API_KEY=EMPTY` yourself, since the placeholder below is applied only to an endpoint this cell resolved to a private address.


In [ ]:
import ipaddress
import json
import os
import re
import shlex
import shutil
import socket
import subprocess
import sys
import time
import urllib.error
import urllib.request
from pathlib import Path
from urllib.parse import urlsplit


if not isinstance(NEMOCLAW_INFERENCE_PROXY, bool):
    raise TypeError("NEMOCLAW_INFERENCE_PROXY must be True or False")


def _resolved_addresses(host):
    try:
        return sorted(
            {
                info[4][0]
                for info in socket.getaddrinfo(host, 443, type=socket.SOCK_STREAM)
            }
        )
    except socket.gaierror as exc:
        print(f"Could not resolve {host}; leaving the endpoint unchanged: {exc}", flush=True)
        return []


def _proxy_port_is_open():
    try:
        with socket.create_connection(("127.0.0.1", INFERENCE_API_PROXY_PORT), timeout=0.5):
            return True
    except OSError:
        return False


def _probe_local_vllm_port(port):
    """Classify localhost:<port> as ("free" | "serving" | "busy", detail).

    NemoClaw chooses between installing managed vLLM and attaching to a running
    one by curling http://127.0.0.1:<port>/v1/models (2s connect, 5s total), so
    ask the same question the same way. "serving" means it will attach; "busy" is
    a listener that answers nothing usable there, which NemoClaw reads as
    absent — it will install vLLM and the container will fail to bind the port.
    """
    try:
        with socket.create_connection(("127.0.0.1", port), timeout=2):
            pass
    except OSError:
        return "free", ""
    url = f"http://127.0.0.1:{port}/v1/models"
    try:
        with urllib.request.urlopen(url, timeout=5) as response:
            payload = json.loads(response.read(65536) or b"{}")
    except urllib.error.HTTPError as exc:
        # A model server with auth on still counts as one: NemoClaw probes a
        # managed vLLM's /health, which needs no key, and attaches to it.
        if exc.code in (401, 403):
            return "serving", f"HTTP {exc.code}, authentication required"
        return "busy", f"HTTP {exc.code} {exc.reason}"
    except Exception as exc:
        return "busy", f"{type(exc).__name__}: {exc}"
    models = [
        entry.get("id", "")
        for entry in (payload.get("data") or [])
        if isinstance(entry, dict) and entry.get("id")
    ]
    if models:
        return "serving", "model " + ", ".join(models)
    return "serving", "answers on /v1/models but listed no model, so it may not be vLLM"


def _ensure_inference_api_proxy(upstream_host):
    if _proxy_port_is_open():
        print(f"Inference API proxy already listening on port {INFERENCE_API_PROXY_PORT}.", flush=True)
        return
    if not INFERENCE_API_PROXY_PATH.is_file():
        raise FileNotFoundError(
            f"NemoClaw needs the inference API proxy, but it was not found at {INFERENCE_API_PROXY_PATH}. "
            "Set INFERENCE_API_PROXY_PATH to inference-api-proxy.py."
        )

    proxy_env = os.environ.copy()
    proxy_env["INFERENCE_API_UPSTREAM"] = upstream_host
    proxy_env["INFERENCE_API_PROXY_HOST"] = "0.0.0.0"
    proxy_env["INFERENCE_API_PROXY_PORT"] = str(INFERENCE_API_PROXY_PORT)
    log_path = Path("/tmp/inference-api-proxy.log")
    with log_path.open("ab") as log_file:
        process = subprocess.Popen(
            [sys.executable, str(INFERENCE_API_PROXY_PATH)],
            env=proxy_env,
            stdout=log_file,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )

    for _ in range(50):
        if _proxy_port_is_open():
            print(
                f"Started {INFERENCE_API_PROXY_PATH} on port {INFERENCE_API_PROXY_PORT} "
                f"-> {upstream_host} (log: {log_path}).",
                flush=True,
            )
            return
        if process.poll() is not None:
            break
        time.sleep(0.1)
    raise RuntimeError(f"Inference API proxy failed to start; inspect {log_path}")


# Only the `custom` provider takes an endpoint URL from this notebook, and NemoClaw
# rejects one whose host resolves to a private address (SSRF guard) — exactly what a
# self-hosted server looks like. Resolve the host so the checks below can spot that
# case: it routes through the host proxy, and it is also the case where a blank
# COMPATIBLE_API_KEY is fine. `_effective_nemoclaw_endpoint_url` is the URL onboard
# actually receives: the configured one, or the proxy's if it starts.
_effective_nemoclaw_endpoint_url = NEMOCLAW_ENDPOINT_URL
_inference_proxy_active = False
_endpoint = urlsplit(NEMOCLAW_ENDPOINT_URL) if NEMOCLAW_ENDPOINT_URL else None
_endpoint_host = _endpoint.hostname if _endpoint else None
_endpoint_addresses = []
_endpoint_is_local = False
if NEMOCLAW_PROVIDER == "custom" and _endpoint_host:
    _endpoint_addresses = _resolved_addresses(_endpoint_host)
    _endpoint_is_local = any(
        not ipaddress.ip_address(address).is_global for address in _endpoint_addresses
    )
if _endpoint_is_local and NEMOCLAW_INFERENCE_PROXY:
    print(
        f"{_endpoint_host} resolves to {_endpoint_addresses}; "
        "using the host proxy to avoid NemoClaw SSRF rejection.",
        flush=True,
    )
    _ensure_inference_api_proxy(_endpoint_host)
    _endpoint_path = _endpoint.path.rstrip("/") or "/v1"
    _effective_nemoclaw_endpoint_url = (
        f"http://host.openshell.internal:{INFERENCE_API_PROXY_PORT}{_endpoint_path}"
    )
    _inference_proxy_active = True

# Fail fast on a provider the section 1.2 cells left half-configured.
if NEMOCLAW_PROVIDER == "custom":
    if not NEMOCLAW_ENDPOINT_URL:
        raise RuntimeError(
            'NEMOCLAW_PROVIDER is "custom", which needs NEMOCLAW_ENDPOINT_URL — the URL of the '
            "OpenAI-compatible server to use. Set it in the (a) cell."
        )
    if not NEMOCLAW_MODEL:
        raise RuntimeError(
            'NEMOCLAW_PROVIDER is "custom", which needs NEMOCLAW_MODEL — a bare endpoint URL does '
            "not say what to request. Set it in the (a) cell."
        )
    if not COMPATIBLE_API_KEY:
        if _endpoint_is_local:
            # The installer rejects a blank key, but a self-hosted server ignores the value,
            # so there is nothing for the user to supply here.
            COMPATIBLE_API_KEY = "EMPTY"
            print(f'COMPATIBLE_API_KEY was blank; sending the "EMPTY" placeholder to {_endpoint_host}.')
        else:
            raise RuntimeError(
                f"COMPATIBLE_API_KEY is required to reach {_endpoint_host}. "
                "Set it in the (a) cell — a public endpoint needs its real bearer token."
            )
else:
    # Only `custom` consumes NEMOCLAW_ENDPOINT_URL, so any other provider paired with one
    # would quietly ignore the server the user pointed at — surface that as a conflict.
    if NEMOCLAW_ENDPOINT_URL:
        raise RuntimeError(
            f"NEMOCLAW_ENDPOINT_URL is set but NEMOCLAW_PROVIDER is {NEMOCLAW_PROVIDER!r}, which ignores it.\n"
            '  - to use that endpoint: set NEMOCLAW_PROVIDER = "custom"\n'
            f"  - to use {NEMOCLAW_PROVIDER!r}: clear NEMOCLAW_ENDPOINT_URL"
        )
    if NEMOCLAW_PROVIDER == "build" and not NVIDIA_API_KEY:
        raise RuntimeError(
            "No agent provider configured. Run ONE of the section 1.2 cells:\n"
            "  - (a) OpenAI-compatible endpoint: needs NEMOCLAW_ENDPOINT_URL, NEMOCLAW_MODEL, COMPATIBLE_API_KEY\n"
            "  - (b) NemoClaw-managed local model: needs NEMOCLAW_PROVIDER, e.g. install-vllm\n"
            "  - (c) build.nvidia.com: needs NVIDIA_API_KEY"
        )

# NemoClaw reads its configuration from environment variables.
env = os.environ.copy()
env["VSS_REPO_DIR"] = str(VSS_REPO_DIR)
env["NEMOCLAW_SANDBOX_NAME"] = NEMOCLAW_SANDBOX_NAME
env["NEMOCLAW_INSTALL_REF"] = NEMOCLAW_INSTALL_REF
env["NEMOCLAW_PROVIDER"] = NEMOCLAW_PROVIDER
if NEMOCLAW_PROVIDER not in ("build", "custom"):
    # Gates the local-server providers off DGX Spark/Station; a no-op where it is not needed.
    env["NEMOCLAW_EXPERIMENTAL"] = "1"
env["NEMOCLAW_NON_INTERACTIVE"] = "1"
env["NEMOCLAW_ACCEPT_THIRD_PARTY_SOFTWARE"] = "1"
env["NEMOCLAW_AGENT"] = AGENT_RUNTIME
env["NVIDIA_API_KEY"] = NVIDIA_API_KEY
_vllm_model = NEMOCLAW_MODEL if NEMOCLAW_PROVIDER == "install-vllm" else ""
for _model_key, _model_value in (("NEMOCLAW_MODEL", NEMOCLAW_MODEL), ("NEMOCLAW_VLLM_MODEL", _vllm_model)):
    if _model_value:
        env[_model_key] = _model_value
    else:
        # Drop, not skip: os.environ.update(env) below would leave the last run's slug winning.
        env.pop(_model_key, None)
        os.environ.pop(_model_key, None)
env["NEMOCLAW_VLLM_PORT"] = str(NEMOCLAW_VLLM_PORT)
HF_TOKEN = str(globals().get("HF_TOKEN", "")).strip()
if HF_TOKEN:
    # NemoClaw hands this to the one-shot weight downloader (`hf download`); it is not
    # baked into the long-running vLLM container or kept in onboarding state.
    env["HF_TOKEN"] = HF_TOKEN
    print("HF_TOKEN set: Hugging Face downloads will be authenticated.")
if NEMOCLAW_PROVIDER == "custom":
    env["NEMOCLAW_ENDPOINT_URL"] = _effective_nemoclaw_endpoint_url
    env["COMPATIBLE_API_KEY"] = COMPATIBLE_API_KEY
    print(
        "Custom OpenAI-compatible provider: "
        f"{_effective_nemoclaw_endpoint_url} model={NEMOCLAW_MODEL} "
        f"key set={bool(COMPATIBLE_API_KEY)}"
    )
else:
    # These belong to `custom` alone. Drop any inherited from the shell or from an
    # earlier custom run of this cell — `os.environ.update(env)` below keeps them for
    # the rest of the kernel, and a stale endpoint would reach the installer here.
    for _custom_only in ("NEMOCLAW_ENDPOINT_URL", "COMPATIBLE_API_KEY"):
        env.pop(_custom_only, None)
        os.environ.pop(_custom_only, None)
# CHAT_UI_URL must be correct at onboard time: the sandbox derives the UI
# allowed origins and the 0.0.0.0 forward bind from it (gateway.* is read-only later).
_chat_fqdn = brev_secure_link_fqdn(AGENT_DASHBOARD_PORT)
if _chat_fqdn:
    env["CHAT_UI_URL"] = f"https://{_chat_fqdn}"
    print("CHAT_UI_URL:", env["CHAT_UI_URL"])
else:
    print(
        f"WARNING: no FQDN for port {AGENT_DASHBOARD_PORT} in {BREV_ENVIRONMENT_CONTEXT_PATH}; "
        "onboard will proceed without baking a remote UI origin."
    )
os.environ.update(env)  # export for the ! commands below


def _version_of(text):
    match = re.search(r"v?(\d+\.\d+\.\d+)", text or "")
    return match.group(1) if match else None


install_ref = NEMOCLAW_INSTALL_REF.strip()
if not install_ref:
    raise RuntimeError("NEMOCLAW_INSTALL_REF must not be empty.")
install_url = f"https://raw.githubusercontent.com/NVIDIA/NemoClaw/{install_ref}/install.sh"


#### Install CLI and onboard the sandbox

Puts `~/.local/bin` on `PATH`, installs the pinned NemoClaw / NemoHermes CLI when the requested ref is not already present, then runs non-interactive onboard if the sandbox does not exist — or `onboard --recreate-sandbox` when `NEMOCLAW_RECREATE_SANDBOX` is set. Onboard builds the sandbox image from the harness's Dockerfile (`.openclaw/Dockerfile` or `.hermes/Dockerfile`) with `--from`, using the Dockerfile's directory as the build context, per NemoClaw's custom-image workflow; the image carries the VSS skills, the workspace docs and the `vss` CLI. Takes several minutes whenever it onboards.

<span style="color:red"><strong>This cell replaces an existing sandbox</strong> whenever <code>NEMOCLAW_RECREATE_SANDBOX</code> is <code>True</code>, the default in section 1.3. The sandbox and its agent sessions are discarded, and 3.2–3.4 reinstall the policy, skills, and workspace docs. Set it to <code>False</code> in 1.3 to keep an existing sandbox.</span>


In [ ]:
local_bin = HOME_DIR / ".local" / "bin"
if str(local_bin) not in os.environ.get("PATH", "").split(os.pathsep):
    os.environ["PATH"] = f"{local_bin}{os.pathsep}{os.environ.get('PATH', '')}"

_session_path = HOME_DIR / ".nemoclaw" / "onboard-session.json"

installed = None
if shutil.which(AGENT_CLI):
    _ver = !{AGENT_CLI} --version 2>&1
    installed = _version_of(" ".join(_ver))

if installed and installed == _version_of(install_ref):
    print(f"{AGENT_LABEL} {installed} already installed, skipping installer.", flush=True)
else:
    print(f"Installing {AGENT_LABEL} {install_ref} (takes a few minutes)...", flush=True)
    !cd ~ && bash -o pipefail -c "curl -fsSL {install_url} | bash"
    install_exit_code = _exit_code
    if install_exit_code != 0:
        if not shutil.which(AGENT_CLI):
            raise AssertionError(f"{AGENT_LABEL} install failed")
        print(
            f"{AGENT_LABEL} installer exited non-zero after installing {AGENT_CLI}; "
            "continuing with explicit onboarding.",
            flush=True,
        )

!{AGENT_CLI} --version
assert _exit_code == 0, f"{AGENT_CLI} is not runnable after install"

!openshell sandbox get {NEMOCLAW_SANDBOX_NAME} >/dev/null 2>&1
_sandbox_exists = _exit_code == 0

# Onboard is the only step that applies the provider, model, endpoint, API key,
# AGENT_RUNTIME, and vLLM GPU, and it cannot reconfigure a sandbox that already
# exists — so a settings change needs the sandbox replaced.
_recreate_sandbox = _sandbox_exists and NEMOCLAW_RECREATE_SANDBOX
if _recreate_sandbox:
    print(
        "NEMOCLAW_RECREATE_SANDBOX is set — onboarding with the settings above; it "
        f"discards sandbox {NEMOCLAW_SANDBOX_NAME!r} and everything in it once it "
        "reaches the sandbox phase...",
        flush=True,
    )

# NemoClaw resolves the provider while onboarding: a model server already
# answering on NEMOCLAW_VLLM_PORT makes it attach to that one ('vllm') rather
# than install its own, and it then refuses --vllm-gpu-device and aborts the run.
_vllm_gpu_flag = AGENT_VLLM_GPU_FLAG
if NEMOCLAW_PROVIDER == "install-vllm":
    _vllm_port_state, _vllm_port_detail = _probe_local_vllm_port(NEMOCLAW_VLLM_PORT)
    if _vllm_port_state == "busy":
        raise RuntimeError(
            f"localhost:{NEMOCLAW_VLLM_PORT} is held by something that serves no model "
            f"({_vllm_port_detail}). Free the port, point NEMOCLAW_VLLM_PORT at a free "
            "one, or wait and re-run if a vLLM there is still loading its weights."
        )
    if _vllm_port_state == "serving" and _vllm_gpu_flag:
        print(
            f"vLLM already serving on localhost:{NEMOCLAW_VLLM_PORT} ({_vllm_port_detail}) — NemoClaw will "
            f"attach to it, so NEMOCLAW_VLLM_GPU_DEVICE={NEMOCLAW_VLLM_GPU_DEVICE!r} is dropped and that "
            "server keeps the GPU it started on. Remove it (`docker rm -f nemoclaw-vllm` for a NemoClaw-managed "
            "one) to install vLLM on the requested GPU instead.",
            flush=True,
        )
        _vllm_gpu_flag = ""

# Onboard reads its settings from the environment, not its flags, so log both.
# Keys are derived: whatever differs from the pristine SHELL_ENV, plus NEMOCLAW_*.
_SECRET_ENV_NAME = re.compile(r"KEY|TOKEN|SECRET|PASSWORD|CREDENTIAL", re.IGNORECASE)


def _onboard_env_keys():
    keys = {key for key in os.environ if key.startswith("NEMOCLAW_")}
    keys |= {
        key
        for key in set(os.environ) | set(SHELL_ENV)
        if os.environ.get(key) != SHELL_ENV.get(key)
    }
    return sorted(keys)


def _echo_onboard_cmd(cmd):
    print("Onboard environment:", flush=True)
    for _key in _onboard_env_keys():
        _value = os.environ.get(_key)
        if _value is None:
            # Present in the shell that started the kernel, dropped since.
            _shown = "(unset)"
        elif _SECRET_ENV_NAME.search(_key):
            _shown = f"(set, {len(_value)} chars)" if _value else "(empty)"
        elif len(_value) > 120:
            _shown = f"{_value[:117]}..."
        else:
            _shown = _value
        print(f"  {_key}={_shown}", flush=True)
    print(f"$ {cmd}", flush=True)


# Custom sandbox image: NemoClaw builds it from the Dockerfile with `--from`, the
# Dockerfile's parent directory as the build context (NEMOCLAW_FROM_DOCKERFILE is
# the env spelling for non-interactive onboard; NEMOCLAW_SANDBOX_NAME is already
# exported, which --from requires so it cannot clobber the default sandbox).
if not AGENT_IMAGE_DOCKERFILE.is_file():
    raise FileNotFoundError(f"AGENT_IMAGE_DOCKERFILE not found: {AGENT_IMAGE_DOCKERFILE}")
os.environ["NEMOCLAW_FROM_DOCKERFILE"] = str(AGENT_IMAGE_DOCKERFILE)
_from_flag = f" --from {shlex.quote(str(AGENT_IMAGE_DOCKERFILE))}"

if not _sandbox_exists or _recreate_sandbox:
    _recreate_flag = (" --recreate-sandbox" if _recreate_sandbox else "") + _from_flag
    # Checked only for the flag this run really passes -- not for one the port
    # probe dropped, and not when the sandbox already stands and no onboard
    # follows. `nemoclaw` rejects a bad device too, but minutes into the run.
    if _vllm_gpu_flag:
        orchestrator_mcp_helper.require_gpu_device(
            "NEMOCLAW_VLLM_GPU_DEVICE",
            NEMOCLAW_VLLM_GPU_DEVICE,
            remedy="Fix it in section 1.3, or leave it blank for the provider default (all visible GPUs).",
        )
    print(
        f"Onboarding sandbox {NEMOCLAW_SANDBOX_NAME!r} "
        f"(agent={AGENT_RUNTIME}, takes several minutes)...",
        flush=True,
    )
    _onboard_cmd = f"cd ~ && {AGENT_ONBOARD_CMD}{_recreate_flag}{_vllm_gpu_flag}"
    _echo_onboard_cmd(_onboard_cmd)
    !{_onboard_cmd}
    if _exit_code != 0 and AGENT_RETRY_ONBOARD_ON_FAIL:
        print(f"{AGENT_LABEL} onboard failed; retrying with refreshed sandbox base-image resolution...", flush=True)
        os.environ.pop("NEMOCLAW_HERMES_SANDBOX_BASE_IMAGE_REF", None)
        os.environ["NEMOCLAW_SANDBOX_BASE_IMAGE_REFRESH"] = "1"
        _echo_onboard_cmd(_onboard_cmd)
        !{_onboard_cmd}
    if _exit_code != 0 and AGENT_RETRY_ONBOARD_ON_FAIL:
        print(f"{AGENT_LABEL} onboard still failed; retrying with --fresh to discard the failed onboarding session...", flush=True)
        _onboard_cmd = f"cd ~ && {AGENT_ONBOARD_CMD} --fresh{_recreate_flag}{_vllm_gpu_flag}"
        _echo_onboard_cmd(_onboard_cmd)
        !{_onboard_cmd}
    assert _exit_code == 0, f"{AGENT_CLI} onboard failed"

    if _session_path.is_file():
        _session = json.loads(_session_path.read_text())
        print("Onboarded provider:", _session.get("provider") or "(unrecorded)")
        print("Onboarded model:", _session.get("model") or "(unrecorded)")
print(f"Sandbox {NEMOCLAW_SANDBOX_NAME!r} ready.", flush=True)


### 3.2 Apply the VSS sandbox policy and deployment origin

Merges `assets/vss_nemoclaw_policy.yaml` into the base OpenShell policy, then, when `VSS_PUBLIC_URL` is set or structured HITL is explicitly enabled, uploads a rendered `ENV.md` over the copy baked into the sandbox image. The render fills the deployment origin when present and records `HITL_ENABLED`. The checked-in image defaults to an empty origin and launch-safe `HITL_ENABLED=false`, so the agent asks for the origin and other follow-ups in ordinary chat.

The policy is rendered on the way in: the `VSS_PUBLIC_URL` set in 1.3 supplies the host and port of the `vss-k8s-ingress` egress entry, since a Kubernetes deployment's Ingress differs per cluster. Left empty, the shipped placeholder host stays — it resolves nowhere, which is the right answer for a Compose sandbox and a `403` on the first call for a Kubernetes one.


In [ ]:
import re
import tempfile
from urllib.parse import urlsplit

# Matches the host/port pair of the vss-k8s-ingress entry, and only that one:
# the block key anchors it, and the lazy span stops at the first `- host:`
# after it.
_POLICY_K8S_HOST = re.compile(r"(  vss-k8s-ingress:\n(?:.*\n)*?\s*- host: )\S+(\n\s*port: )\d+")
# Every allowed_ips list, matched by its own indentation so the replacement
# keeps the block's shape.
_POLICY_ALLOWED_IPS = re.compile(r"(?m)^(?P<indent> +)allowed_ips:\n(?:(?P=indent)  - \S+\n)+")


def validated_origin(value: str) -> str:
    """Return `value` as `scheme://host:port`, or raise ValueError saying why not.

    This cell writes the result into a policy YAML entry and, below, into ENV.md's
    `export` line, which the sandbox agent sources -- so `http://h\"$(id)`,
    which clears every other check urlsplit affords, is command substitution
    by the time the agent starts. A bare origin is the one shape that cannot
    carry shell or YAML syntax into either file. The port is filled from the
    scheme when absent: `vss configure` records the origin verbatim, and the
    Elasticsearch client rejects a URL without one.
    """
    if not value:
        return ""
    origin = urlsplit(value)
    if origin.scheme not in ("http", "https"):
        raise ValueError(f"VSS_PUBLIC_URL needs an http:// or https:// scheme, got {value!r}")
    if origin.path or origin.query or origin.fragment or origin.username or origin.password:
        raise ValueError(f"VSS_PUBLIC_URL must be a bare origin -- no path, query, or credentials: {value!r}")
    # urlsplit lowercases the host, so this is every character a DNS name may
    # hold. IPv6 literals fail it too, which suits hosts that are nip.io names.
    if not origin.hostname or not re.fullmatch(r"[a-z0-9.-]+", origin.hostname):
        raise ValueError(f"VSS_PUBLIC_URL has no usable hostname: {value!r}")
    port = origin.port or (443 if origin.scheme == "https" else 80)  # .port raises on a bad one
    return f"{origin.scheme}://{origin.hostname}:{port}"


def _fill_k8s_ingress(text: str, source: Path) -> str:
    """Fill the Ingress egress entry in *text* from VSS_PUBLIC_URL.

    The policy ships a placeholder host rather than a real cluster, so an
    unfilled entry allowlists nothing. Leaving it that way is a supported
    state: a Compose sandbox never calls an Ingress, and a Kubernetes one
    reports the resulting `CONNECT tunnel failed, response 403` instead of
    silently reaching a host nobody chose.
    """
    if not VSS_PUBLIC_URL:
        print("vss-k8s-ingress: unset (VSS_PUBLIC_URL empty) — placeholder host kept", flush=True)
        return text
    origin = urlsplit(VSS_PUBLIC_URL)  # validated above, so the port is set
    rendered, filled = _POLICY_K8S_HOST.subn(
        rf"\g<1>{origin.hostname}\g<2>{origin.port}",
        text,
        count=1,
    )
    if not filled:
        raise RuntimeError(f"No vss-k8s-ingress host entry to fill in {source}")
    print(f"vss-k8s-ingress: {origin.hostname}:{origin.port}", flush=True)
    return rendered


def _narrow_allowed_ips(text: str, source: Path) -> str:
    """Replace a policy's guessed bridge CIDRs with the sandbox's own.

    A policy file has to name every range Docker might have handed OpenShell,
    because a hand-applied one has nothing to detect from. Here the sandbox
    exists (3.1 onboarded it), so host.openshell.internal resolves to the
    gateway of the networks it is actually on and the list collapses to those.
    Detection failing is not fatal: the file's own ranges still cover it.
    """
    cidrs = orchestrator_mcp_helper.sandbox_host_cidrs(NEMOCLAW_SANDBOX_NAME)
    if not cidrs:
        print(f"allowed_ips ({source.name}): sandbox networks not detected — file's CIDRs kept",
              flush=True)
        return text

    def _rewrite(match: re.Match) -> str:
        indent = match.group("indent")
        entries = "".join(f"{indent}  - {cidr}\n" for cidr in cidrs)
        return f"{indent}allowed_ips:\n{entries}"

    rendered, filled = _POLICY_ALLOWED_IPS.subn(_rewrite, text)
    print(f"allowed_ips ({source.name}): {', '.join(cidrs)} ({filled} entries)", flush=True)
    return rendered


def _render_policy(policy: Path, staging: Path, *, fill_ingress: bool = True) -> Path:
    """Narrow the bridge CIDRs, fill the Ingress host, return what to apply.

    Every policy this cell applies goes through here, so a generated preset
    gets the same detected CIDRs as the shipped file. `fill_ingress=False`
    suits one that carries no vss-k8s-ingress entry to fill.
    """
    source = policy.read_text(encoding="utf-8")
    rendered = _fill_k8s_ingress(source, policy) if fill_ingress else source
    rendered = _narrow_allowed_ips(rendered, policy)
    if rendered == source:
        return policy
    out = staging / policy.name
    out.write_text(rendered, encoding="utf-8")
    return out


VSS_PUBLIC_URL = validated_origin(VSS_PUBLIC_URL)
if not POLICY_PATH.is_file():
    raise FileNotFoundError(f"Missing policy file: {POLICY_PATH}")
with tempfile.TemporaryDirectory(prefix="vss-policy-") as _staging_dir:
    _policy = _render_policy(POLICY_PATH, Path(_staging_dir))
    _policy_add_cmd = AGENT_POLICY_ADD_CMD.format(sandbox=NEMOCLAW_SANDBOX_NAME, path=_policy)
    !{_policy_add_cmd}
assert _exit_code == 0, "policy add failed"

# Optional model-router egress, written by deploy_vss_switchyard.ipynb when the
# operator routes the agent's LLM traffic. Applied here so it is part of sandbox
# setup and is reapplied after a re-onboard, which discards the live policy.
# Absent on every deployment that does not route, which is the common case.
MODEL_ROUTER_POLICY = Path(
    os.environ.get("MODEL_ROUTER_POLICY",
                   HOME_DIR / "vss-model-routing" / "vss_model_router_policy.yaml")
)
if MODEL_ROUTER_POLICY.is_file():
    # Narrow the router preset's allowed_ips as well: a sandbox on a bridge
    # network they do not list cannot reach the router.
    with tempfile.TemporaryDirectory(prefix="vss-router-policy-") as _staging_dir:
        _router_policy = _render_policy(
            MODEL_ROUTER_POLICY, Path(_staging_dir), fill_ingress=False
        )
        _router_add_cmd = AGENT_POLICY_ADD_CMD.format(
            sandbox=NEMOCLAW_SANDBOX_NAME, path=_router_policy
        )
        !{_router_add_cmd}
    assert _exit_code == 0, f"router policy add failed: {MODEL_ROUTER_POLICY}"
    print(f"Applied model-router egress from {MODEL_ROUTER_POLICY}", flush=True)


# Deployment settings for the agent. The image already carries an empty origin
# and HITL=false, so upload an override only when the origin is set or structured
# HITL is explicitly enabled.
if VSS_PUBLIC_URL or HITL_ENABLED:
    _env_md = WORKSPACE_DIR / f"_{WORKSPACE_VARIANT}" / "ENV.md"
    if not _env_md.is_file():
        _env_md = WORKSPACE_DIR / "ENV.md"
    if not _env_md.is_file():
        raise FileNotFoundError(f"No ENV.md under {WORKSPACE_DIR} to render VSS_PUBLIC_URL into")
    _text = _env_md.read_text(encoding="utf-8")
    _rendered = _text
    if VSS_PUBLIC_URL:
        _rendered, _origin_filled = re.subn(
            r"^export VSS_PUBLIC_URL=.*$", f'export VSS_PUBLIC_URL="{VSS_PUBLIC_URL}"',
            _rendered, count=1, flags=re.MULTILINE,
        )
        if not _origin_filled:
            raise RuntimeError(f"No 'export VSS_PUBLIC_URL=' line to fill in {_env_md}")
    _rendered, _hitl_filled = re.subn(
        r"^export HITL_ENABLED=.*$",
        f'export HITL_ENABLED={str(HITL_ENABLED).lower()}',
        _rendered, count=1, flags=re.MULTILINE,
    )
    if not _hitl_filled:
        raise RuntimeError(f"No 'export HITL_ENABLED=' line to fill in {_env_md}")
    with tempfile.TemporaryDirectory(prefix="vss-env-md-") as _staging_dir:
        _out = Path(_staging_dir) / "ENV.md"
        _out.write_text(_rendered, encoding="utf-8")
        # dest is a DIRECTORY: the OpenShell transport does mkdir + tar-extract into it.
        !openshell sandbox exec -n {NEMOCLAW_SANDBOX_NAME} -- sh -c "mkdir -p {WORKSPACE_REMOTE_DIR}"
        _upload_cmd = AGENT_UPLOAD_CMD.format(sandbox=NEMOCLAW_SANDBOX_NAME, doc=_out, dest=WORKSPACE_REMOTE_DIR)
        !{_upload_cmd} >/dev/null
        assert _exit_code == 0, "ENV.md upload failed"
    print(f"ENV.md: VSS_PUBLIC_URL={VSS_PUBLIC_URL or '(empty)'}, HITL_ENABLED={str(HITL_ENABLED).lower()} uploaded to {WORKSPACE_REMOTE_DIR}", flush=True)


### 3.3 Register the VSS Orchestrator MCP

With `ORCHESTRATOR_ENABLE_HTTPS = False` (the default) this section is a no-op: the agent uses the locally deployed orchestrator MCP served by `deploy_vss_orchestrator.ipynb` (sections 3.3–4), so no registration is needed here. Set `ORCHESTRATOR_ENABLE_HTTPS = True` in both notebooks only if you want to register an HTTPS MCP endpoint with the sandbox instead.

When HTTPS is enabled, registers the host-side VSS Orchestrator MCP (`ORCHESTRATOR_MCP_SERVER`) at `ORCHESTRATOR_MCP_URL` — the `HOST_INTERNAL_ALIAS` address the sandbox can actually reach over `https`. Uses the selected harness MCP grammar (`AGENT_MCP_*_CMD`). Skipped if already registered, so if you flip the scheme after a first run, remove the old entry first (`AGENT_MCP_REMOVE_CMD` with that server name) before re-running.


In [ ]:
if not ORCHESTRATOR_ENABLE_HTTPS:
    print(
        "ORCHESTRATOR_ENABLE_HTTPS is False — skipping VSS Orchestrator MCP registration.",
        flush=True,
    )
else:
    _mcp_status_cmd = AGENT_MCP_STATUS_CMD.format(
        sandbox=NEMOCLAW_SANDBOX_NAME, server=ORCHESTRATOR_MCP_SERVER
    )
    !{_mcp_status_cmd} >/dev/null 2>&1
    if _exit_code == 0:
        print(f"MCP server {ORCHESTRATOR_MCP_SERVER!r} already registered.", flush=True)
    else:
        _mcp_add_cmd = AGENT_MCP_ADD_CMD.format(
            sandbox=NEMOCLAW_SANDBOX_NAME, server=ORCHESTRATOR_MCP_SERVER, url=ORCHESTRATOR_MCP_URL
        )
        !{_mcp_add_cmd}
        assert _exit_code == 0, "mcp add failed"


### 3.4 Configure optional agent webhooks

Writes harness-specific webhook config, then restarts the gateway so it takes effect. If the managed restart hits `SUPERVISOR_UNAVAILABLE`, falls back to `AGENT_RECOVER_CMD`. Skipped when `AGENT_HOOKS_ENABLED` is off.

- **OpenClaw** (`AGENT_RUNTIME=openclaw`): `hooks.enabled` / `hooks.path` / `hooks.token` on the dashboard port.
- **Hermes** (`AGENT_RUNTIME=hermes`): `platforms.webhook.*` in `/sandbox/.hermes/config.yaml` (port `AGENT_WEBHOOK_PORT`, default `8644`), then an OpenShell forward for that port. Hermes `gateway-token` is unrelated (API bearer on `:8642`).


In [ ]:
import subprocess
import time


# `gateway restart` and `recover` can exit non-zero with SUPERVISOR_UNAVAILABLE
# when their managed-control handshake gives up before the replacement gateway
# finishes booting, so ask the gateway itself before treating that as fatal.
def agent_gateway_healthy(timeout_s: int = 90, poll_s: int = 3, attempt_s: int = 15) -> bool:
    health_url = f"http://127.0.0.1:{AGENT_DASHBOARD_PORT}/health"
    # curl already reports 000 when it cannot connect, so no `||` fallback here.
    probe = f'curl -s -o /dev/null -w "%{{http_code}}" --max-time 5 {health_url}'
    deadline = time.monotonic() + timeout_s
    while True:
        try:
            # `sandbox exec` can wedge before it ever spawns curl, so each attempt
            # needs its own ceiling for the deadline below to mean anything.
            probed = subprocess.run(
                ["openshell", "sandbox", "exec", "-n", NEMOCLAW_SANDBOX_NAME, "--", "sh", "-c", probe],
                capture_output=True,
                text=True,
                timeout=attempt_s,
            )
            code = (probed.stdout or "").strip()
        except subprocess.TimeoutExpired:
            code = ""
        # 401/403 still prove the gateway is listening behind its token.
        if code.startswith("2") or code in ("401", "403"):
            return True
        if time.monotonic() >= deadline:
            return False
        time.sleep(poll_s)


# Write webhook keys first, then restart separately. `config set --restart` can
# update the config and still exit non-zero with SUPERVISOR_UNAVAILABLE when the
# in-sandbox supervisor is gone; recover relaunches it.
# - OpenClaw: hooks.* (gateway.* is off-limits; UI origins come from CHAT_UI_URL).
# - Hermes: platforms.webhook.* (HMAC secret; listens on AGENT_WEBHOOK_PORT).
config_sets = []
if AGENT_HOOKS_ENABLED:
    if AGENT_RUNTIME == "openclaw":
        config_sets.append(("hooks.enabled", "true"))
        config_sets.append(("hooks.path", AGENT_HOOKS_PATH or "/hooks"))
        if AGENT_HOOKS_TOKEN:
            config_sets.append(("hooks.token", AGENT_HOOKS_TOKEN))
    elif AGENT_RUNTIME == "hermes":
        if not AGENT_HOOKS_TOKEN:
            raise RuntimeError("AGENT_HOOKS_TOKEN is required to configure Hermes webhooks.")
        config_sets.extend(
            [
                ("platforms.webhook.enabled", "true"),
                ("platforms.webhook.extra.port", str(AGENT_WEBHOOK_PORT)),
                ("platforms.webhook.extra.secret", AGENT_HOOKS_TOKEN),
                # Known route for callers; secret matches the global fallback.
                ("platforms.webhook.extra.routes.vss-notebook.secret", AGENT_HOOKS_TOKEN),
                ("platforms.webhook.extra.routes.vss-notebook.deliver", "log"),
                (
                    "platforms.webhook.extra.routes.vss-notebook.prompt",
                    "VSS notebook webhook: {__raw__}",
                ),
            ]
        )

if not config_sets:
    reason = (
        "webhooks disabled"
        if not AGENT_HOOKS_ENABLED
        else f"no webhook config for AGENT_RUNTIME={AGENT_RUNTIME!r}"
    )
    print(f"No sandbox config changes needed ({reason}).", flush=True)
else:
    for _key, _value in config_sets:
        _quoted = shlex.quote(_value)  # values may contain spaces/quotes
        _config_set_cmd = AGENT_CONFIG_SET_CMD.format(
            sandbox=NEMOCLAW_SANDBOX_NAME, key=_key, value=_quoted
        )
        !{_config_set_cmd}
        assert _exit_code == 0, f"config set failed: {_key}"
    print("Restarting gateway to apply webhook config...", flush=True)
    _gateway_restart_cmd = AGENT_GATEWAY_RESTART_CMD.format(sandbox=NEMOCLAW_SANDBOX_NAME)
    !{_gateway_restart_cmd}
    if _exit_code != 0:
        print("Managed gateway restart failed — falling back to sandbox recover...", flush=True)
        _recover_cmd = AGENT_RECOVER_CMD.format(sandbox=NEMOCLAW_SANDBOX_NAME)
        !{_recover_cmd}
        if _exit_code != 0:
            print("Recover reported failure — probing the gateway directly...", flush=True)
            assert agent_gateway_healthy(), "gateway is down after webhook config"
            print("Gateway is answering its health probe; continuing.", flush=True)
    print("Sandbox config applied; gateway restarted.", flush=True)

    if AGENT_RUNTIME == "hermes":
        # Manifest only forwards 18789/8642; webhook adapter needs its own forward.
        _wh_port = str(AGENT_WEBHOOK_PORT)
        !openshell forward stop {_wh_port} {NEMOCLAW_SANDBOX_NAME} >/dev/null 2>&1
        !openshell forward start --background {_wh_port} {NEMOCLAW_SANDBOX_NAME}
        assert _exit_code == 0, f"openshell forward start failed for webhook port {_wh_port}"
        print(
            f"Hermes webhook listening (forwarded): "
            f"http://127.0.0.1:{_wh_port}/health  "
            f"and routes at http://127.0.0.1:{_wh_port}/webhooks/<name> "
            f"(e.g. /webhooks/vss-notebook).",
            flush=True,
        )


### 3.5 Open the Agent UI

Open the **Agent UI** for the sandbox so you can chat with the agent and drive VSS from it.

- **OpenClaw** (`ui_uses_gateway_token`): prints the dashboard origin with a `#token=` fragment from `AGENT_GATEWAY_TOKEN_CMD`.
- **Hermes** (`dashboard_url_cmd`): prints `nemohermes <sandbox> dashboard-url` (plain URL; on Brev the host is rewritten to the secure-link FQDN). Hermes `gateway-token` is the OpenAI-compatible API bearer on port `8642`, not the UI token.

Run the next cell to check the dashboard forward, re-bind it if onboard's copy died (rebuild/recover) or is bound loopback-only, print a fresh **Agent UI** link, then open it in your browser. Only this sandbox's own forward is touched: the process holding the port must carry the sandbox id, so an unrelated service on `AGENT_DASHBOARD_PORT` stops the cell with an error instead of being adopted or killed.

<span style="color:red"><strong>Not on Brev?</strong> If you are accessing the UI from a different machine, open an SSH tunnel before opening the Agent web UI from below:<br/><code>ssh -L &lt;AGENT_DASHBOARD_PORT&gt;:127.0.0.1:&lt;AGENT_DASHBOARD_PORT&gt; &lt;user&gt;@&lt;agent-host&gt;</code><br/></span>


In [ ]:
import os
import re
import shlex
import subprocess
import time
from urllib.parse import urlparse, urlunparse


def fetch_gateway_token():
    if not AGENT_GATEWAY_TOKEN_CMD:
        raise RuntimeError(f"{AGENT_LABEL} has no gateway_token_cmd in AGENT_HARNESS_PROFILES")
    cmd = shlex.split(AGENT_GATEWAY_TOKEN_CMD.format(sandbox=NEMOCLAW_SANDBOX_NAME))
    result = subprocess.run(cmd, capture_output=True, text=True, check=True)
    return result.stdout.strip()


def fetch_dashboard_url():
    if not AGENT_DASHBOARD_URL_CMD:
        raise RuntimeError(f"{AGENT_LABEL} has no dashboard_url_cmd in AGENT_HARNESS_PROFILES")
    cmd = shlex.split(AGENT_DASHBOARD_URL_CMD.format(sandbox=NEMOCLAW_SANDBOX_NAME))
    result = subprocess.run(cmd, capture_output=True, text=True, check=True)
    return result.stdout.strip()


gateway_container = resolve_openshell_gateway_container(NEMOCLAW_SANDBOX_NAME)
print("Gateway container:", gateway_container)

if not gateway_container:
    raise RuntimeError("Could not determine the OpenShell gateway container; no UI link generated.")

_chat_fqdn = brev_secure_link_fqdn(AGENT_DASHBOARD_PORT)
# No FQDN has two causes that want opposite binds: not on Brev, where loopback plus
# the SSH tunnel below is right, or on Brev with a context file this user cannot read
# (/etc/brev is 0700 root:root), where loopback is unreachable for the secure-link
# edge and for the UI container alike. BREV_ENV_ID is in the environment, so it still
# names Brev when the file cannot be read.
_brev_env_id = brev_environment_id()
_remote_bind = bool(_chat_fqdn or _brev_env_id)

# Onboard starts the dashboard forward, but it dies with the kernel or terminal that
# launched it and a rebuild/recover drops it; the UI then 503s. Re-establish it here.
_health = f"http://127.0.0.1:{AGENT_DASHBOARD_PORT}/health"


def _dashboard_forward_rows():
    """(BIND, PID, STATUS) rows OpenShell records for this sandbox's dashboard port."""
    listing = subprocess.run(["openshell", "forward", "list"], capture_output=True, text=True)
    rows = []
    for line in re.sub(r"\x1b\[[0-9;]*m", "", listing.stdout).splitlines():
        cols = line.split()
        if (
            len(cols) >= 5
            and cols[0] == NEMOCLAW_SANDBOX_NAME
            and cols[2] == str(AGENT_DASHBOARD_PORT)
        ):
            rows.append((cols[1], cols[3], cols[4]))
    return rows


def _port_listener_pids():
    """PIDs listening on the dashboard port; empty when lsof is unavailable."""
    try:
        listing = subprocess.run(
            ["lsof", "-t", f"-i:{AGENT_DASHBOARD_PORT}", "-sTCP:LISTEN"], capture_output=True, text=True
        )
    except FileNotFoundError:
        return []
    return listing.stdout.split()


def _pid_args(pid):
    """Command line of *pid*, empty when it is already gone."""
    return subprocess.run(["ps", "-p", pid, "-o", "args="], capture_output=True, text=True).stdout.strip()


def _dashboard_forward_holder():
    """(BIND, PID) of this sandbox's running forward that holds the port, or None."""
    # A row can read running after its process is gone and its PID be recycled, so the
    # listener's own command line, never the PID alone, decides whether it is ours.
    holders = set(_port_listener_pids())
    for bind, pid, status in _dashboard_forward_rows():
        if status == "running" and pid in holders and _is_sandbox_forward(_pid_args(pid)):
            return bind, pid
    return None


def _openshell_sandbox_describe(sandbox):
    """`openshell sandbox get` output, or raise when the host gateway cannot answer."""
    got = subprocess.run(["openshell", "sandbox", "get", sandbox], capture_output=True, text=True)
    if got.returncode != 0:
        detail = (got.stderr or got.stdout or "").strip() or f"exit {got.returncode}"
        raise RuntimeError(
            f"OpenShell cannot describe sandbox {sandbox!r}:\n{detail}\n"
            "This section needs the host-side OpenShell gateway; bring it back, then re-run."
        )
    return re.sub(r"\x1b\[[0-9;]*m", "", got.stdout)


def _openshell_sandbox_id(sandbox):
    """Sandbox id for *sandbox*, or None when the description carries no Id."""
    found = re.search(r"^\s*Id:\s+(\S+)", _openshell_sandbox_describe(sandbox), re.MULTILINE)
    return found.group(1) if found else None


_sandbox_id = _openshell_sandbox_id(NEMOCLAW_SANDBOX_NAME)


def _is_sandbox_forward(args):
    """True when a command line is this sandbox's forward. Without an id nothing is ours,
    so a foreign listener is never adopted and never killed."""
    return bool(_sandbox_id and re.search(rf"--sandbox-id[=\s]+{re.escape(_sandbox_id)}\b", args))

# A forward bound to 127.0.0.1 answers this health probe and still 503s behind the
# Brev secure link, which reaches the port from off-loopback, so the bind is checked
# too. 0.0.0.0 covers loopback, so it is never re-bound the other way. A healthy probe
# says nothing about who holds the port, so this sandbox must own a running forward.
_held = _dashboard_forward_holder()
_healthy = subprocess.run(["curl", "-fsS", "-o", "/dev/null", _health], capture_output=True).returncode == 0
# The port can change hands around the probe, so the same PID must still hold it after.
_bind = _held[0] if _held and _dashboard_forward_holder() == _held else None
if _healthy and _bind and (_bind == "0.0.0.0" or not _remote_bind):
    print(f"Dashboard forward already up on {_bind}:{AGENT_DASHBOARD_PORT}")
else:
    if _healthy and _bind:
        _edge = _chat_fqdn or f"Brev environment {_brev_env_id}"
        print(f"Dashboard forward is bound {_bind}, which the {_edge} edge cannot reach; re-binding.")
    elif _healthy:
        print(f"Port {AGENT_DASHBOARD_PORT} answers but no running forward belongs to {NEMOCLAW_SANDBOX_NAME!r}; re-establishing.")
    # A dead row or an orphaned ssh child can still hold the port and make `forward
    # start` fail, so clear it first. Only this sandbox's own forward may be killed.
    subprocess.run(
        ["openshell", "forward", "stop", str(AGENT_DASHBOARD_PORT), NEMOCLAW_SANDBOX_NAME],
        check=False, capture_output=True, text=True,
    )
    for _pid in _port_listener_pids():
        _args = _pid_args(_pid)
        if not _is_sandbox_forward(_args):
            raise RuntimeError(
                f"Port {AGENT_DASHBOARD_PORT} is held by pid {_pid} ({_args}), which is not "
                f"{NEMOCLAW_SANDBOX_NAME!r}'s dashboard forward; stop it before continuing."
            )
        print(f"Clearing stale forward on {AGENT_DASHBOARD_PORT} (pid {_pid}): {_args}")
        subprocess.run(["kill", _pid])
    # setsid detaches the forward from this kernel's session so a kernel restart does
    # not take it down. A remote (non-loopback) UI origin needs 0.0.0.0, not 127.0.0.1.
    _fwd = f"0.0.0.0:{AGENT_DASHBOARD_PORT}" if _remote_bind else str(AGENT_DASHBOARD_PORT)
    !setsid -f openshell forward start --background {_fwd} {NEMOCLAW_SANDBOX_NAME}
    # setsid returns before the forward is up, and a healthy probe can come from an
    # unrelated listener, so readiness means one PID of ours holds the port across the
    # probe.
    for _ in range(15):
        _held = _dashboard_forward_holder()
        _answers = _held and subprocess.run(
            ["curl", "-fsS", "-o", "/dev/null", _health], capture_output=True
        ).returncode == 0
        if _answers and _dashboard_forward_holder() == _held:
            print("Dashboard forward ready on", _fwd)
            break
        time.sleep(1)
    else:
        print(f"WARNING: {_health} is not answered by {NEMOCLAW_SANDBOX_NAME!r}'s forward; check `nemoclaw {NEMOCLAW_SANDBOX_NAME} status`.")

if _chat_fqdn:
    origin = f"https://{_chat_fqdn}"
else:
    origin = f"http://127.0.0.1:{AGENT_DASHBOARD_PORT}"
    agent_host = subprocess.run(
        ["hostname", "-I"], capture_output=True, text=True, check=True
    ).stdout.split()[0]
    ssh_user = os.environ.get("USER", "ubuntu")
    RED, RESET = "\033[31m", "\033[0m"
    print(f"{RED}Make sure an SSH tunnel is running on your laptop before opening the Agent web UI:{RESET}")
    print(f"{RED}  $ ssh -L {AGENT_DASHBOARD_PORT}:localhost:{AGENT_DASHBOARD_PORT} {ssh_user}@{agent_host}{RESET}")
    if BREV_ENVIRONMENT_CONTEXT_PATH:
        print(
            f"{RED}No FQDN for port {AGENT_DASHBOARD_PORT} in {BREV_ENVIRONMENT_CONTEXT_PATH}; "
            f"using localhost tunnel URL instead.{RESET}"
        )
    if _brev_env_id:
        print(
            f"{RED}Brev environment {_brev_env_id} is set, so that file was unreadable as "
            f"{ssh_user} and via `sudo -n`, or does not publish this port; the forward is "
            f"bound 0.0.0.0 either way. Grant {ssh_user} passwordless sudo, or point "
            f"BREV_ENVIRONMENT_CONTEXT_PATH at a copy it can read, to get the secure-link "
            f"URL back.{RESET}"
        )

if AGENT_UI_USES_GATEWAY_TOKEN:
    # OpenClaw control UI authenticates via #token= from `sandbox gateway token`.
    token = fetch_gateway_token()
    agent_ui_url = f"{origin}/#token={token}" if token else origin
elif AGENT_DASHBOARD_URL_CMD:
    # Hermes: `dashboard-url` (plain URL). `gateway-token` is the :8642 API bearer — not for the UI.
    raw = fetch_dashboard_url()
    if _chat_fqdn and raw:
        parsed = urlparse(raw)
        agent_ui_url = urlunparse(
            ("https", _chat_fqdn, parsed.path or "", "", parsed.query, parsed.fragment)
        )
    else:
        agent_ui_url = raw or origin
else:
    agent_ui_url = origin

print("Sandbox:", NEMOCLAW_SANDBOX_NAME)
print("Agent UI:", agent_ui_url)
if AGENT_CONNECT_CMD:
    print(f"{AGENT_LABEL} terminal: {AGENT_CONNECT_CMD.format(sandbox=NEMOCLAW_SANDBOX_NAME)}")


### 3.6 [OPTIONAL] Verify sandbox, policy, workspace, and optional webhooks

The next cell checks:

- whether the sandbox exists,
- the current active sandbox policy metadata,
- the expected local policy path and version,
- whether agent webhooks are healthy when enabled (OpenClaw: `POST /hooks/agent`; Hermes: `GET :AGENT_WEBHOOK_PORT/health`),
- the installed OpenClaw skills/workspace files or Hermes top-level workspace docs.


In [ ]:
from datetime import datetime, timezone
import json
import re
import shlex
import subprocess
from pathlib import Path


def run(cmd, check=False, echo=True):
    print("$", shlex.join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True, check=check)
    if echo:
        if r.stdout:
            print(r.stdout)
        if r.stderr:
            print(r.stderr)
    return r


print("Verification time (UTC):", datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S %Z"))
print("Sandbox:", NEMOCLAW_SANDBOX_NAME)
print("Expected policy file:", POLICY_PATH)

policy_text = Path(POLICY_PATH).read_text()
preset_name_match = re.search(r"^\s+name:\s*(\S+)", policy_text, re.MULTILINE)
print("Expected preset name:", preset_name_match.group(1) if preset_name_match else "unknown")

sandbox_result = run(["openshell", "sandbox", "get", NEMOCLAW_SANDBOX_NAME], echo=False)
sandbox_summary = "\n".join(part for part in (sandbox_result.stdout, sandbox_result.stderr) if part)
sandbox_summary = re.sub(r"\x1b\[[0-9;]*m", "", sandbox_summary)
phase_match = re.search(r"Phase:\s+(.+)", sandbox_summary)
namespace_match = re.search(r"Namespace:\s+(.+)", sandbox_summary)
sandbox_id_match = re.search(r"Id:\s+(.+)", sandbox_summary)
print("Sandbox namespace:", namespace_match.group(1).strip() if namespace_match else "unknown")
print("Sandbox phase:", phase_match.group(1).strip() if phase_match else "unknown")
print("Sandbox id:", sandbox_id_match.group(1).strip() if sandbox_id_match else "unknown")

policy_result = run(["openshell", "policy", "get", NEMOCLAW_SANDBOX_NAME])

policy_summary = "\n".join(part for part in (policy_result.stdout, policy_result.stderr) if part)
status_match = re.search(r"Status:\s+(.+)", policy_summary)
active_match = re.search(r"Active:\s+(.+)", policy_summary)
hash_match = re.search(r"Hash:\s+([0-9a-f]+)", policy_summary)
print("Active policy status:", status_match.group(1).strip() if status_match else "unknown")
print("Active policy version:", active_match.group(1).strip() if active_match else "unknown")
print("Active policy hash:", hash_match.group(1) if hash_match else "unknown")

gateway_container = resolve_openshell_gateway_container(NEMOCLAW_SANDBOX_NAME)
print("Gateway container:", gateway_container)

if not WORKSPACE_DIR.is_dir():
    raise FileNotFoundError(f"Missing workspace source dir: {WORKSPACE_DIR}")
expected_workspace_md = tuple(sorted(p.name for p in WORKSPACE_DIR.glob("*.md")))
if not expected_workspace_md:
    raise RuntimeError(f"No .md files found in {WORKSPACE_DIR}; cannot verify workspace install.")
print("Expected workspace .md files:", list(expected_workspace_md))

if AGENT_HOOKS_ENABLED:
    if AGENT_RUNTIME == "openclaw":
        if not AGENT_HOOKS_TOKEN:
            raise RuntimeError("AGENT_HOOKS_TOKEN is required to verify OpenClaw hooks.")

        hooks_path = "/" + AGENT_HOOKS_PATH.strip("/")
        hooks_url = f"http://127.0.0.1:{AGENT_DASHBOARD_PORT}{hooks_path}/agent"
        hooks_payload = json.dumps(
            {
                "name": f"{AGENT_LABEL} notebook verification",
                "message": "test",
            }
        )
        hooks_cmd = [
            "curl",
            "-sS",
            "-w", "\n%{http_code}",  # append the HTTP status as the last stdout line
            "-X",
            "POST",
            hooks_url,
            "-H",
            f"Authorization: Bearer {AGENT_HOOKS_TOKEN}",
            "-H",
            "Content-Type: application/json",
            "-d",
            hooks_payload,
        ]
        hooks_result = subprocess.run(hooks_cmd, capture_output=True, text=True)
        hooks_body, _, hooks_status = (hooks_result.stdout or "").rpartition("\n")
        hooks_body, hooks_status = hooks_body.strip(), hooks_status.strip()
        if hooks_result.returncode != 0:
            # Only a transport failure exits non-zero here; 4xx still exits 0.
            print(
                f"{AGENT_LABEL} hooks test: FAIL — could not reach {hooks_url}"
                " — run section 3.4 to start the dashboard forward"
            )
        elif hooks_status == "200":
            print(f"{AGENT_LABEL} hooks test: PASS (HTTP 200)")
        else:
            hints = {
                "404": " — run section 3.4 to configure webhooks",
                "401": " — token mismatch; re-run section 3.4 to apply the current AGENT_HOOKS_TOKEN",
            }
            hint = hints.get(hooks_status, "")
            print(f"{AGENT_LABEL} hooks test: FAIL (HTTP {hooks_status or 'unknown'}){hint}")
    elif AGENT_RUNTIME == "hermes":
        health_url = f"http://127.0.0.1:{AGENT_WEBHOOK_PORT}/health"
        health = subprocess.run(["curl", "-fsS", health_url], capture_output=True, text=True)
        body = (health.stdout or "").strip()
        if health.returncode == 0 and '"status"' in body and "ok" in body:
            print(f"{AGENT_LABEL} webhook health: PASS ({health_url} -> {body})")
            print(f"  route example: http://127.0.0.1:{AGENT_WEBHOOK_PORT}/webhooks/vss-notebook")
        else:
            detail = body or (health.stderr or "").strip() or f"exit {health.returncode}"
            print(f"{AGENT_LABEL} webhook health: FAIL ({health_url}) — {detail}")
            print("  — run section 3.4 to configure Hermes webhooks / port forward")
    else:
        print(f"Agent hooks test skipped: no webhook verify for AGENT_RUNTIME={AGENT_RUNTIME!r}.")
else:
    print("Agent hooks test skipped: AGENT_HOOKS_ENABLED is false.")

if gateway_container:
    def _sandbox_exec(sandbox_cmd):
        return [
            "openshell", "sandbox", "exec", "-n", NEMOCLAW_SANDBOX_NAME, "--",
            "sh", "-lc", sandbox_cmd,
        ]

    def _in_sandbox_show(label, sandbox_cmd):
        full = _sandbox_exec(sandbox_cmd)
        print(f"\n=== {label} ===")
        print("$", shlex.join(full))
        r = subprocess.run(full, capture_output=True, text=True, stdin=subprocess.DEVNULL)
        if r.stdout:
            print(r.stdout)
        if r.stderr:
            print(r.stderr)
        return r

    if AGENT_VERIFY_KIND == "sandbox_docs":
        for _label_tmpl, _cmd_tmpl in AGENT_VERIFY_CMDS:
            _verify_fmt = dict(
                label=AGENT_LABEL,
                workspace_remote_dir=WORKSPACE_REMOTE_DIR,
                scheme=MCP_SCHEME,
                host_alias=HOST_INTERNAL_ALIAS,
                mcp_port=MCP_PORT,
            )
            _in_sandbox_show(
                _label_tmpl.format(**_verify_fmt),
                _cmd_tmpl.format(**_verify_fmt),
            )
    elif AGENT_VERIFY_KIND == "openclaw_workspace":
        # _in_sandbox_show("openclaw plugins list", "openclaw plugins list")
        _in_sandbox_show("openclaw plugins doctor", "openclaw plugins doctor")

        skills_proc = subprocess.run(
            _sandbox_exec("openclaw skills list --json"),
            capture_output=True, text=True, stdin=subprocess.DEVNULL,
        )
        workspace_dir = None
        print("\n=== openclaw skills (non-bundled) ===")
        try:
            payload = json.loads(skills_proc.stdout)
            skills = payload["skills"] if isinstance(payload, dict) else payload
            if isinstance(payload, dict):
                workspace_dir = payload.get("workspaceDir")
            non_bundled = [s for s in skills if not s.get("bundled", False)]
            if non_bundled:
                for s in non_bundled:
                    print(f"- {s['name']}")
            else:
                print("No non-bundled OpenClaw skills found.")
        except (json.JSONDecodeError, TypeError) as exc:
            print(f"Could not parse skills list JSON: {exc}")
            if skills_proc.stderr:
                print(skills_proc.stderr)

        print(f"\n=== workspace .md files in {workspace_dir or '<unknown>'} ===")
        if workspace_dir:
            ls_cmd = f"ls -1 {shlex.quote(workspace_dir)} 2>/dev/null"
            ls_proc = subprocess.run(_sandbox_exec(ls_cmd), capture_output=True, text=True, stdin=subprocess.DEVNULL)
            present = {line.strip() for line in ls_proc.stdout.splitlines() if line.strip().endswith(".md")}
            for name in expected_workspace_md:
                mark = "OK  " if name in present else "MISS"
                print(f"  {mark}  {name}")
            extra = present - set(expected_workspace_md)
            if extra:
                print(f"  (also present: {sorted(extra)})")
        else:
            print("  workspaceDir not found in skills JSON payload; check skipped.")
    else:
        print(f"No workspace verification handler for verify_kind={AGENT_VERIFY_KIND!r}.")
else:
    print("Could not determine the OpenShell gateway container; runtime-specific workspace checks were skipped.")
